# MEC v1.4 — Final Policy Freeze

Development-only. Hidden seeds are not generated, loaded, or evaluated here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
assert Path('/content/drive/MyDrive/MEC_V1_3_R2_CHECKPOINTS').exists(), 'Missing R2 checkpoints'
assert Path('/content/drive/MyDrive/MEC_V1_4_R1_CHECKPOINTS').exists(), 'Missing v1.4 checkpoints'
print('Drive prerequisites PASS')

In [ ]:
!pip -q install scikit-learn==1.6.1 scipy==1.16.3
import sklearn, scipy
print('sklearn',sklearn.__version__,'scipy',scipy.__version__)

In [ ]:
from pathlib import Path
SRC=Path('/content/mec_final_src'); SRC.mkdir(parents=True,exist_ok=True)
(SRC/'MEC_E2E_V1_FROZEN_CONFIG.json').write_text('{\n  "protocol_id": "MEC_E2E_V1_FROZEN_2026_08_24",\n  "status": "FROZEN_BEFORE_EXECUTION",\n  "freeze_date": "2026-08-24",\n  "purpose": "End-to-end learned-world-model mechanism diagnosis and intervention-readiness test",\n  "environments": [\n    {\n      "id": "Reacher-v5",\n      "role": "primary",\n      "state_interface": "actual single-system observed joint q, qdot, action; no paired nominal-system internals",\n      "train_transitions": 60000,\n      "calibration_transitions": 20000,\n      "nominal_test_transitions": 20000\n    },\n    {\n      "id": "InvertedDoublePendulum-v5",\n      "role": "confirmatory_cross_family",\n      "state_interface": "actual single-system observed generalized coordinates/velocities and action; no paired nominal-system internals",\n      "train_transitions": 80000,\n      "calibration_transitions": 20000,\n      "nominal_test_transitions": 20000\n    }\n  ],\n  "world_models": {\n    "count_per_environment": 8,\n    "seeds": [\n      681101,\n      681102,\n      681103,\n      681104,\n      681105,\n      681106,\n      681107,\n      681108\n    ],\n    "architecture": "Residual MLP, 3 hidden layers x 128, SiLU",\n    "target": "wrapped coordinate delta + velocity delta",\n    "epochs": 120,\n    "batch_size": 512,\n    "optimizer": "AdamW",\n    "learning_rate": 0.002,\n    "weight_decay": 1e-05,\n    "quality_gate": {\n      "models_required": 7,\n      "of": 8,\n      "aggregate_state_rmse_vs_persistence_ratio_max": 0.5,\n      "normalized_residual_target_mse_max": 0.25\n    }\n  },\n  "split_seeds": {\n    "nominal_train": 682001,\n    "calibration": 682002,\n    "development_baseline_fit": 682003,\n    "hidden_test": [\n      947213,\n      947227,\n      947261\n    ]\n  },\n  "hidden_test": {\n    "pure_fault_episodes_per_class_per_seed": 200,\n    "combined_or_oov_episodes_per_seed": 200,\n    "trajectory_length_reacher": 40,\n    "trajectory_length_inverted_double_pendulum": 80,\n    "randomized_factors": [\n      "initial pose",\n      "initial velocity",\n      "control trajectory",\n      "joint damping",\n      "body/link mass within declared train/test ranges",\n      "fault magnitude",\n      "fault sign",\n      "observation nuisance",\n      "actuator nuisance"\n    ],\n    "pure_fault_classes": [\n      "ENCODER_OFFSET",\n      "MECHANICAL_REFERENCE_SHIFT",\n      "DYNAMICS_DAMPING_FAULT",\n      "ACTUATOR_BIAS"\n    ],\n    "combined_oov_examples": [\n      "ENCODER_OFFSET + DYNAMICS_DAMPING_FAULT",\n      "MECHANICAL_REFERENCE_SHIFT + ACTUATOR_BIAS",\n      "outside-training-hull morphology"\n    ],\n    "generic_static_fault_direction": "uniform angle on [0,2pi)",\n    "development_and_hidden_parameter_ranges_identical_except_oov_morphology": true\n  },\n  "allowed_diagnostic_input": [\n    "residuals r_t = y_{t+1} - f_hat(y_t,u_t) from the evaluated learned world model",\n    "world-model predictive ensemble disagreement if available from independent seeds",\n    "declared physical reference measurements acquired by the diagnostic policy",\n    "actual actions and timestamps",\n    "calibration statistics frozen from nominal held-out residuals"\n  ],\n  "forbidden_diagnostic_input": [\n    "difference between a nominal MuJoCo instance and a faulty MuJoCo instance",\n    "true fault label",\n    "hidden MuJoCo model parameters",\n    "direct qacc difference between nominal and fault simulators",\n    "test-set-specific threshold tuning",\n    "analytic injected Gaussian noise as the sole evidence of empirical calibration"\n  ],\n  "evidence_stages": {\n    "E0": "passive same-channel learned-model residual history only",\n    "E1": "E0 + one rank-1 independent reference direction",\n    "E2": "E0 + full-rank independent reference for static mechanism split",\n    "D2": "learned-model residuals after two frozen diagnostic operating points",\n    "D3": "learned-model residuals after three affinely independent frozen diagnostic operating points"\n  },\n  "uncertainty": {\n    "primary": "empirical calibration from held-out nominal learned-model residuals",\n    "method": "split-conformal/selective threshold on intervention risk; covariance/whitening reported descriptively",\n    "gaussian_theory": "secondary diagnostic only; never used as empirical validation unless held-out residual diagnostics support it"\n  },\n  "actions": [\n    "RECALIBRATE_SENSOR",\n    "SERVICE_MECHANICAL_REFERENCE",\n    "SERVICE_DYNAMICS",\n    "SERVICE_ACTUATOR",\n    "ABSTAIN_ACQUIRE_MORE_EVIDENCE",\n    "ESCALATE_MULTI_OR_OOV"\n  ],\n  "baselines": [\n    "PASSIVE_RF_RESIDUAL_CLASSIFIER",\n    "PASSIVE_MLP_RESIDUAL_CLASSIFIER",\n    "RANDOM_PROBE_EQUAL_BUDGET",\n    "ACTIVE_MODEL_DISCRIMINATION_MAX_PREDICTIVE_SEPARATION",\n    "FULL_EVIDENCE_ORACLE_UPPER_BOUND"\n  ],\n  "primary_metrics": [\n    "wrong intervention risk among committed cases",\n    "commitment coverage",\n    "risk-coverage curve and AURC",\n    "correct intervention rate",\n    "mean diagnostic evidence/probe cost to safe commitment",\n    "combined/OOV abstention-or-escalation rate"\n  ],\n  "sealed_gates": {\n    "P0_world_model_quality": ">=7/8 models per environment meet both frozen quality thresholds",\n    "P1_static_below_threshold_ambiguity": "on matched encoder-vs-mechanical pure faults, best generic held-out classifier AUC <=0.65 under E0 and E1",\n    "P2_static_full_rank_resolution": "under E2, MEC/structured correct intervention >=0.90, wrong-intervention risk <=0.05, coverage >=0.80",\n    "P3_dynamic_below_threshold_ambiguity": "damping-vs-actuator-nuisance classifier AUC <=0.65 under D2",\n    "P4_dynamic_threshold_resolution": "under D3, correct intervention >=0.90, wrong-intervention risk <=0.05, coverage >=0.80",\n    "P5_selective_end_to_end": "on all pure hidden faults, MEC wrong-intervention risk <=0.05 at coverage >=0.70",\n    "P6_ood_safety": "combined/OOV wrong pure-intervention rate <=0.05 and abstain/escalate rate >=0.80",\n    "P7_evidence_efficiency": "at matched <=5% wrong-intervention risk, MEC uses <=0.85x mean diagnostic probe cost of RANDOM_PROBE_EQUAL_BUDGET; report comparison to active model discrimination without requiring superiority",\n    "P8_cross_family": "P2/P4/P5 directional conclusions hold in both environment families; if confirmatory environment fails, no cross-family robotics claim",\n    "P9_seed_stability": "no more than 1 of 8 world-model seeds may violate P5 by >2 percentage points; all three hidden-test seeds are reported",\n    "P10_no_posthoc": "all seeds, splits, thresholds, fault ranges, metrics and gates remain unchanged after first hidden unseal"\n  },\n  "decision_rule": {\n    "PASS_STRONG": "P0-P10 all true",\n    "PARTIAL_NARROW": "core theory-aligned structural gates pass but one or more generalization/baseline/safety gates fail; narrow paper to passed scope",\n    "FAIL_END_TO_END": "P1/P2/P3/P4/P5 fails materially; do not claim MEC works end-to-end for learned world models"\n  },\n  "reporting": {\n    "confidence_intervals": "paired bootstrap 95% CI over hidden scenarios, stratified by mechanism and model seed",\n    "must_report": [\n      "all hidden seeds",\n      "all 8 model seeds",\n      "all pure and combined/OOV classes",\n      "all baselines",\n      "failures and abstentions",\n      "per-environment and pooled results"\n    ],\n    "bootstrap_replicates": 5000,\n    "auc_ci": "DeLong or paired bootstrap; use paired bootstrap if implementation dependency unavailable"\n  },\n  "data_generation": {\n    "action_policy_train": "piecewise-constant random controls; segment length uniformly 1..5 steps",\n    "action_fraction_of_control_range_train": [\n      -0.8,\n      0.8\n    ],\n    "action_fraction_of_control_range_hidden": [\n      -1.0,\n      1.0\n    ],\n    "development_fault_episodes_per_class_per_environment": 120,\n    "development_combined_oov_episodes_per_environment": 120,\n    "development_seed": 682003\n  },\n  "fault_ranges": {\n    "encoder_offset_rad_abs": [\n      0.08,\n      0.25\n    ],\n    "mechanical_reference_shift_rad_abs": [\n      0.08,\n      0.25\n    ],\n    "damping_multiplier": [\n      1.5,\n      3.0\n    ],\n    "actuator_bias_fraction_of_ctrl_range_abs": [\n      0.08,\n      0.25\n    ],\n    "fault_sign": "independent symmetric +/- unless vector direction is explicitly constructed",\n    "static_threshold_pair": {\n      "E1_reference_direction": [\n        1.0,\n        0.0\n      ],\n      "matched_fault_direction": [\n        0.0,\n        1.0\n      ],\n      "E2_reference_matrix": [\n        [\n          1.0,\n          0.0\n        ],\n        [\n          0.0,\n          1.0\n        ]\n      ],\n      "note": "adversarial threshold arm only; generic hidden static faults use uniformly random 2D direction"\n    },\n    "dynamic_threshold_points_reacher": {\n      "pose_q": [\n        0.15,\n        -0.22\n      ],\n      "velocity_points_D2": [\n        [\n          0.0,\n          0.0\n        ],\n        [\n          0.7,\n          0.0\n        ]\n      ],\n      "velocity_point_added_D3": [\n        0.0,\n        0.7\n      ],\n      "control": [\n        0.0,\n        0.0\n      ]\n    },\n    "dynamic_threshold_points_inverted_double_pendulum": {\n      "hinge_pose_q": [\n        0.05,\n        -0.05\n      ],\n      "hinge_velocity_points_D2": [\n        [\n          0.0,\n          0.0\n        ],\n        [\n          0.5,\n          0.0\n        ]\n      ],\n      "hinge_velocity_point_added_D3": [\n        0.0,\n        0.5\n      ],\n      "cart_state": "0 position, 0 velocity unless Gymnasium health constraints require epsilon-safe deterministic adjustment logged before any hidden fault labels are read",\n      "control": [\n        0.0\n      ]\n    }\n  },\n  "morphology_and_nuisance_ranges": {\n    "train_mass_multiplier": [\n      0.85,\n      1.15\n    ],\n    "hidden_in_hull_mass_multiplier": [\n      0.8,\n      1.2\n    ],\n    "hidden_oov_mass_multiplier_low": [\n      0.65,\n      0.75\n    ],\n    "hidden_oov_mass_multiplier_high": [\n      1.25,\n      1.4\n    ],\n    "train_nominal_damping_multiplier": [\n      0.9,\n      1.1\n    ],\n    "hidden_nominal_nuisance_damping_multiplier": [\n      0.85,\n      1.15\n    ],\n    "sensor_uniform_jitter_q_rad": [\n      -0.002,\n      0.002\n    ],\n    "sensor_uniform_jitter_qdot": [\n      -0.01,\n      0.01\n    ],\n    "note": "the same nuisance generator is present in nominal calibration data; Gaussian trial injection is not part of the primary E2E evaluation"\n  },\n  "diagnostic_method": {\n    "name": "MEC_SELECTIVE_V1",\n    "fit_data": "development fault set only; hidden seeds inaccessible until final evaluation",\n    "fault_detection_score": "whitened learned-world-model residual energy using nominal calibration residual covariance with ridge 1e-6",\n    "static_split": "use learned-model residual history for fault evidence plus declared independent reference measurement; full-rank reference supports mechanical-vs-encoder action split",\n    "dynamic_split": "fit centered affine residual model R(v)=A v+s from learned-model one-step residual summaries at frozen operating points; compare dynamics term ||A|| and intercept term ||s||",\n    "selective_commitment": "mechanism action score must be uniquely maximal and exceed a threshold selected on development/calibration data to target <=5% wrong-intervention risk; otherwise abstain/acquire evidence",\n    "threshold_selection": "choose smallest score threshold on development data whose one-sided Clopper-Pearson 95% upper bound on wrong-intervention risk is <=0.05; if none exists, always abstain for that stage",\n    "no_hidden_tuning": true\n  },\n  "baseline_hyperparameters": {\n    "RF": {\n      "n_estimators": 500,\n      "min_samples_leaf": [\n        2,\n        5,\n        10\n      ],\n      "max_features": [\n        "sqrt",\n        0.5\n      ]\n    },\n    "MLP": {\n      "hidden": [\n        [\n          128,\n          128\n        ],\n        [\n          256,\n          128\n        ]\n      ],\n      "dropout": [\n        0.0,\n        0.1\n      ],\n      "epochs": 100,\n      "early_stopping_patience": 10\n    },\n    "RANDOM_PROBE": {\n      "probe_budget": 3,\n      "seeds": [\n        701,\n        702,\n        703,\n        704,\n        705\n      ]\n    },\n    "ACTIVE_MODEL_DISCRIMINATION": {\n      "candidate_probe_pool_size": 64,\n      "probe_budget": 3,\n      "objective": "maximize minimum pairwise standardized separation between development-fitted mechanism residual predictors",\n      "optimizer": "enumerate frozen candidate pool; deterministic tie break by candidate index"\n    },\n    "ORACLE": {\n      "description": "full allowed evidence stages plus true sufficient evidence allocation; never receives true mechanism label for prediction, only upper-bound probe allocation"\n    }\n  }\n}')
(SRC/'MEC_E2E_V1_4_ROBUST_SET_COMMIT_DEV_FROZEN_CONFIG.json').write_text('{\n  "protocol_id": "MEC_E2E_V1_4_ROBUST_SET_COMMIT_DEV_2026_08_29",\n  "status": "FROZEN_DEVELOPMENT_ONLY_BEFORE_EXECUTION",\n  "predecessor": "MEC_E2E_V1_3_COMPOSITIONAL_VOI_DEV_2026_08_25",\n  "predecessor_disposition": "PRIMARY_FAIL_OOD_COMPOSITION_ONLY",\n  "predecessor_observed_summary": {\n    "wrong_intervention_risk_median": 0.00865817091454273,\n    "coverage_median": 0.9583333333333334,\n    "correct_intervention_rate_median": 0.95,\n    "composite_or_ood_pure_commit_median": 0.13333333333333333,\n    "composite_or_ood_safe_median": 0.8666666666666667,\n    "mec_to_random_probe_cost_ratio_median": 0.6345523591530979,\n    "primary_folds_passed": 1,\n    "efficiency_folds_passed": 8\n  },\n  "scope": "NARROW_REPAIR_OF_PURE_VS_COMPOSITE_OOD_COMMITMENT_ONLY",\n  "environment": "Reacher-v5",\n  "reuse": {\n    "r2_world_model_checkpoint_dir": "/content/drive/MyDrive/MEC_V1_3_R2_CHECKPOINTS",\n    "reuse_world_models": true,\n    "reuse_nominal_residual_normalization": true,\n    "reuse_v13_old_train_feature_records": true,\n    "world_model_retraining_prohibited": true,\n    "theorem_arm": "UNCHANGED_FROM_V1_1",\n    "hidden_unseal": "PROHIBITED"\n  },\n  "development_world_model_seeds": [\n    681101,\n    681102,\n    681103,\n    681104,\n    681105,\n    681106,\n    681107,\n    681108\n  ],\n  "hidden_world_model_seeds_forbidden": [\n    781101,\n    781102,\n    781103,\n    781104,\n    781105,\n    781106,\n    781107,\n    781108\n  ],\n  "hidden_scenario_seeds_forbidden": [\n    957101,\n    957107,\n    957109\n  ],\n  "old_v13_training": {\n    "scenario_seed0": 753001,\n    "group_split_seed": 43117,\n    "use_only_old_train_groups": true\n  },\n  "fresh_v14_qualification": {\n    "scenario_seed0": 764001,\n    "groups": 120,\n    "split_seed": 54217,\n    "calibration_groups": 60,\n    "test_groups": 60,\n    "steps": 40,\n    "fresh_groups_are_never_used_to_fit_base_evidence_heads": true\n  },\n  "head": {\n    "name": "ROBUST_SET_CERTIFICATE_HGB",\n    "mechanism_bits": "four binary HistGradientBoosting heads per evidence stage",\n    "semantic_state": "one 7-state HistGradientBoosting classifier per evidence stage",\n    "semantic_states": [\n      "ENCODER_OFFSET",\n      "MECHANICAL_REFERENCE_SHIFT",\n      "DYNAMICS_DAMPING_FAULT",\n      "ACTUATOR_BIAS",\n      "ENCODER_OFFSET+DYNAMICS_DAMPING_FAULT",\n      "MECHANICAL_REFERENCE_SHIFT+ACTUATOR_BIAS",\n      "OUTSIDE_HULL_MORPHOLOGY"\n    ],\n    "max_iter": 40,\n    "learning_rate": 0.08,\n    "max_leaf_nodes": 15,\n    "l2_regularization": 0.1\n  },\n  "commitment": {\n    "rule": "A pure action is permitted only when the compositional bit heads and semantic-state head jointly certify the same singleton pure mechanism. Otherwise acquire more evidence or escalate.",\n    "certificate_score": "min(top_bit_probability, 1-second_bit_probability, matching_pure_state_probability, 1-nonpure_state_probability)",\n    "calibration": "single scalar certificate threshold per evidence stage",\n    "robustness": "threshold must satisfy one-sided 95% Clopper-Pearson <=5% wrong-pure risk and <=5% nonpure pure-commit rate separately on every source world-model calibration block and pooled source blocks",\n    "selection": "among safe thresholds maximize minimum source-WM pure coverage, then pooled pure coverage",\n    "final_noncertificate": "ESCALATE_MULTI_OR_OOD",\n    "nonfinal_noncertificate": "ACQUIRE_MORE_EVIDENCE"\n  },\n  "probe_policy": {\n    "MEC_VOI": "same v1.3 objective family: predicted mechanism-bit entropy reduction per unit probe cost from passive evidence",\n    "implementation": "lightweight HistGradientBoosting regressors fit only on old v1.3 train groups",\n    "costs": {\n      "static": 2.0,\n      "dynamic": 3.0,\n      "full": 5.0\n    },\n    "random_baseline": "uniform static/dynamic first probe"\n  },\n  "primary_gates": {\n    "wrong_intervention_risk_max": 0.05,\n    "pure_coverage_min": 0.7,\n    "composite_or_ood_pure_commit_max": 0.05,\n    "composite_or_ood_safe_min": 0.95,\n    "fold_passes_required": 6,\n    "of_folds": 8,\n    "median_must_pass": true,\n    "world_model_quality_models_required": 7\n  },\n  "secondary_retention": {\n    "report_v13_reference_wrong_intervention": 0.00865817091454273,\n    "report_v13_reference_coverage": 0.9583333333333334,\n    "report_v13_reference_correct_intervention": 0.95,\n    "report_v13_reference_cost_ratio": 0.6345523591530979,\n    "efficiency_target_ratio_max": 0.85,\n    "efficiency_fold_passes_required": 6,\n    "efficiency_failure_does_not_override_primary": true\n  },\n  "outcome_labels": {\n    "PRIMARY_PASS_EFFICIENCY_PASS": "authorize final v1.4 policy freeze; hidden evaluator may be built next; efficiency claim survives",\n    "PRIMARY_PASS_EFFICIENCY_FAIL": "authorize final v1.4 policy freeze; hidden evaluator may be built next; prohibit efficiency-superiority claim",\n    "PRIMARY_FAIL": "hidden remains sealed; do not tune on v1.4 test groups"\n  },\n  "prohibited": [\n    "hidden world-model seeds",\n    "hidden scenario seeds",\n    "training on held-out world model",\n    "threshold fitting on fresh v1.4 test groups",\n    "retraining world models",\n    "changing v1.1 theorem arm",\n    "rewriting v1.3 failure",\n    "post-hoc relaxation of 5%/70%/95% gates"\n  ]\n}')
(SRC/'MEC_E2E_V1_4_ROBUST_SET_COMMIT_DEV_FROZEN_PROTOCOL.md').write_text("# MEC E2E v1.4 — Robust Set-Valued Commitment Repair\n\n**Protocol ID:** `MEC_E2E_V1_4_ROBUST_SET_COMMIT_DEV_2026_08_29`  \n**Status:** FROZEN DEVELOPMENT-ONLY BEFORE EXECUTION  \n**Hidden status:** SEALED\n\n## Why v1.4 exists\n\nv1.3 R2 failed its preregistered primary gate despite strong pure-fault intervention and probe efficiency. The observed medians were: wrong intervention **0.87%**, coverage **95.83%**, correct intervention **95.0%**, composite/OOD pure commit **13.33%**, composite/OOD safe **86.67%**, and MEC/random probe cost **0.635x**. Therefore v1.4 is not a general architecture rewrite. It is restricted to the final pure-vs-composite/OOD commitment semantics.\n\n## Frozen successes\n\nThe following are not targets for repair and are held fixed in scope: the eight R2 development world models and their nominal residual normalization, the Reacher mechanism/fault generator, the residual evidence representation, the static/dynamic/full probe interface and costs, the v1.1 theorem arm, and the hidden seeds. No world model is retrained.\n\n## Prospective anti-overfit split\n\nThe old v1.3 train groups remain the only data used to fit base evidence heads and the VOI regressors. v1.4 creates **120 fresh development groups** from seed **764001**. A new split seed **54217** produces **60 fresh calibration groups** and **60 fresh qualification-test groups**. Fresh test labels are never used for fitting or threshold selection. Hidden seeds remain inaccessible.\n\n## Repair: robust set-valued pure certificate\n\nEach evidence stage fits four binary mechanism-bit heads and one seven-state semantic classifier using lightweight HistGradientBoosting on the old v1.3 train groups. For candidate pure mechanism `k`, define\n\n`certificate_k = min(p_bit[k], 1 - second_bit_probability, p_state[k], 1 - p_state(nonpure))`.\n\nA pure intervention is allowed only when the highest-scoring mechanism is a singleton certificate above the calibrated stage threshold. Otherwise the policy acquires more evidence (non-final stage) or escalates (full stage). This explicitly prevents the v1.3 failure mode in which a composite case could look like one dominant bit plus one sub-threshold bit.\n\n## Distributionally robust calibration across world models\n\nFor each LOMO fold, calibration uses the **seven source WMs only** on the fresh calibration groups. A candidate threshold is accepted only if its one-sided **95% Clopper-Pearson upper bound** is <=5% for both (a) wrong pure action among committed pure cases and (b) pure-action commits on composite/OOD cases, **for every source WM separately and pooled**. Among safe thresholds, select the threshold maximizing the minimum source-WM pure coverage, then pooled coverage.\n\nThis is intentionally more conservative than v1.3's pooled empirical grid search.\n\n## LOMO qualification\n\nEach fold is evaluated on one held-out development WM and the **fresh v1.4 test groups**. The original scientific bars remain unchanged:\n\n- wrong intervention risk <=5%;\n- pure coverage >=70%;\n- composite/OOD pure commit <=5%;\n- composite/OOD safe >=95%;\n- >=6/8 LOMO folds pass all four;\n- median primary metrics also pass.\n\nEfficiency remains secondary: median MEC/random probe-cost ratio <=0.85 with >=6/8 efficiency-passing folds. Efficiency failure cannot erase a primary scientific pass, but it removes the efficiency-superiority claim.\n\n## Decision\n\n- `PRIMARY_PASS_EFFICIENCY_PASS`: freeze one final v1.4 policy and proceed to build the untouched hidden evaluator.\n- `PRIMARY_PASS_EFFICIENCY_FAIL`: freeze one final v1.4 policy and proceed to hidden, but prohibit an efficiency-superiority claim.\n- `PRIMARY_FAIL`: hidden remains sealed. No threshold tuning against v1.4 test groups is permitted.\n\n## Governance\n\nv1.3's `PRIMARY_FAIL` is permanent historical evidence and is not overwritten. v1.4 is a prospective new development protocol.\n")
(SRC/'mec_e2e_v1_runner.py').write_text('from __future__ import annotations\nimport argparse, json, math, os, random, hashlib\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Dict, List, Tuple, Optional\n\nimport numpy as np\n\n\ndef wrap_angle(x):\n    return (x + np.pi) % (2*np.pi) - np.pi\n\n\ndef seed_all(seed: int):\n    random.seed(seed); np.random.seed(seed)\n    try:\n        import torch\n        torch.manual_seed(seed)\n    except Exception:\n        pass\n\n\ndef sha256_file(p: Path):\n    return hashlib.sha256(p.read_bytes()).hexdigest()\n\n\ndef save_json(p: Path, obj):\n    p.write_text(json.dumps(obj, indent=2))\n\n\n@dataclass\nclass JointMap:\n    qaddr: np.ndarray\n    dof: np.ndarray\n    hinge_local_idx: np.ndarray\n    angle_mask: np.ndarray\n    body_ids: np.ndarray\n\n\nclass EnvAdapter:\n    """Single-system adapter. No paired nominal/fault simulator exists in evaluation methods."""\n    def __init__(self, env_id: str, seed: int):\n        import gymnasium as gym\n        import mujoco\n        self.gym = gym; self.mujoco = mujoco; self.env_id = env_id\n        self.env = gym.make(env_id)\n        self.env.reset(seed=seed)\n        self.model = self.env.unwrapped.model\n        self.data = self.env.unwrapped.data\n        self.jmap = self._joint_map()\n        self.action_low = np.asarray(self.env.action_space.low, dtype=float)\n        self.action_high = np.asarray(self.env.action_space.high, dtype=float)\n        self.base = {\n            \'qpos0\': self.model.qpos0.copy(),\n            \'body_mass\': self.model.body_mass.copy(),\n            \'body_inertia\': self.model.body_inertia.copy(),\n            \'dof_damping\': self.model.dof_damping.copy(),\n        }\n\n    def close(self): self.env.close()\n\n    def _joint_name(self, jid):\n        return self.mujoco.mj_id2name(self.model, self.mujoco.mjtObj.mjOBJ_JOINT, int(jid)) or f\'joint{jid}\'\n\n    def _joint_map(self) -> JointMap:\n        hinge = int(self.mujoco.mjtJoint.mjJNT_HINGE)\n        slide = int(self.mujoco.mjtJoint.mjJNT_SLIDE)\n        jids=[]\n        if self.env_id.startswith(\'Reacher\'):\n            for nm in (\'joint0\',\'joint1\'):\n                jid=self.mujoco.mj_name2id(self.model,self.mujoco.mjtObj.mjOBJ_JOINT,nm)\n                if jid < 0: raise RuntimeError(f\'Missing {nm} in Reacher model\')\n                jids.append(int(jid))\n        else:\n            # Keep scalar dynamic joints; IDP is cart slide + 2 hinge joints.\n            for jid,t in enumerate(self.model.jnt_type):\n                if int(t) in (hinge,slide): jids.append(jid)\n        qaddr=[]; dof=[]; angle=[]; bodies=[]; hinge_local=[]\n        for k,jid in enumerate(jids):\n            qaddr.append(int(self.model.jnt_qposadr[jid])); dof.append(int(self.model.jnt_dofadr[jid]))\n            is_h=int(self.model.jnt_type[jid])==hinge\n            angle.append(is_h); bodies.append(int(self.model.jnt_bodyid[jid]))\n            if is_h: hinge_local.append(k)\n        if len(hinge_local)<2:\n            raise RuntimeError(f\'{self.env_id}: need >=2 hinge joints for frozen MEC protocol; got {len(hinge_local)}\')\n        # first two hinge joints are the diagnostic mechanism coordinates\n        hinge_local=np.asarray(hinge_local[:2],dtype=int)\n        return JointMap(np.asarray(qaddr),np.asarray(dof),hinge_local,np.asarray(angle,dtype=bool),np.asarray(bodies))\n\n    def restore_model(self):\n        self.model.qpos0[:] = self.base[\'qpos0\']\n        self.model.body_mass[:] = self.base[\'body_mass\']\n        self.model.body_inertia[:] = self.base[\'body_inertia\']\n        self.model.dof_damping[:] = self.base[\'dof_damping\']\n        self.mujoco.mj_setConst(self.model, self.data)\n\n    def apply_morphology(self, mass_mult: float, damping_mult: float):\n        # scale unique dynamic bodies associated with selected joints\n        for bid in np.unique(self.jmap.body_ids):\n            if bid == 0: continue\n            self.model.body_mass[bid] = self.base[\'body_mass\'][bid] * mass_mult\n            self.model.body_inertia[bid] = self.base[\'body_inertia\'][bid] * mass_mult\n        self.model.dof_damping[self.jmap.dof] = self.base[\'dof_damping\'][self.jmap.dof] * damping_mult\n        self.mujoco.mj_setConst(self.model, self.data)\n\n    def apply_mechanical_reference(self, h2: np.ndarray):\n        # qpos0 is the MuJoCo joint reference. mj_setConst propagates derived constants.\n        for jj,val in zip(self.jmap.hinge_local_idx, h2):\n            qa=self.jmap.qaddr[jj]\n            self.model.qpos0[qa] = self.base[\'qpos0\'][qa] + float(val)\n        self.mujoco.mj_setConst(self.model, self.data)\n\n    def apply_damping_fault(self, mult: float):\n        # target second diagnostic hinge\n        jj=int(self.jmap.hinge_local_idx[1]); da=self.jmap.dof[jj]\n        self.model.dof_damping[da] = self.base[\'dof_damping\'][da] * mult\n\n    def reset(self, seed:int):\n        self.env.reset(seed=seed)\n\n    def observed_state(self, encoder_offset2=None, jitter_q=0.0, jitter_v=0.0, rng=None):\n        q=np.asarray(self.data.qpos[self.jmap.qaddr]-self.model.qpos0[self.jmap.qaddr],dtype=float).copy()\n        v=np.asarray(self.data.qvel[self.jmap.dof],dtype=float).copy()\n        if encoder_offset2 is not None:\n            for jj,val in zip(self.jmap.hinge_local_idx, encoder_offset2): q[jj]+=float(val)\n        if rng is not None:\n            if jitter_q: q += rng.uniform(-jitter_q,jitter_q,size=len(q))\n            if jitter_v: v += rng.uniform(-jitter_v,jitter_v,size=len(v))\n        return np.r_[q,v]\n\n    def step(self, commanded_action, actuator_bias=None):\n        u=np.asarray(commanded_action,dtype=float).copy()\n        if actuator_bias is not None:\n            b=np.asarray(actuator_bias,dtype=float)\n            if b.size==1: b=np.full_like(u,float(b[0]))\n            u=np.clip(u+b,self.action_low,self.action_high)\n        return self.env.step(u)\n\n    def set_diagnostic_state(self, hinge_q2, hinge_v2, cart_zero=True):\n        # deterministic state setter, never reads hidden fault label/parameters.\n        self.data.qpos[:] = self.model.qpos0\n        self.data.qvel[:] = 0\n        for jj,val in zip(self.jmap.hinge_local_idx,hinge_q2):\n            self.data.qpos[self.jmap.qaddr[jj]] = self.model.qpos0[self.jmap.qaddr[jj]] + float(val)\n        for jj,val in zip(self.jmap.hinge_local_idx,hinge_v2):\n            self.data.qvel[self.jmap.dof[jj]] = float(val)\n        self.data.ctrl[:] = 0\n        self.mujoco.mj_forward(self.model,self.data)\n\n    def static_reference_probe(self, ucmd2, encoder_offset2, R):\n        # Calibration command expressed in nominal numeric coordinate, matching frozen theorem geometry.\n        self.data.qvel[:] = 0\n        for jj,val in zip(self.jmap.hinge_local_idx,ucmd2):\n            qa=self.jmap.qaddr[jj]\n            self.data.qpos[qa] = self.base[\'qpos0\'][qa] + float(val)\n        self.mujoco.mj_forward(self.model,self.data)\n        # physical relative hinge coordinate; sensor coordinate includes encoder offset.\n        phys=np.array([self.data.qpos[self.jmap.qaddr[jj]]-self.model.qpos0[self.jmap.qaddr[jj]] for jj in self.jmap.hinge_local_idx])\n        sens=phys + np.asarray(encoder_offset2,dtype=float)\n        c=sens-np.asarray(ucmd2,dtype=float)\n        w=np.asarray(R,dtype=float)@(phys-np.asarray(ucmd2,dtype=float))\n        return c,w\n\n\ndef feature_map(state: np.ndarray, action: np.ndarray, angle_mask: np.ndarray):\n    state=np.asarray(state); action=np.asarray(action)\n    n=len(angle_mask); q=state[:n]; v=state[n:]\n    fs=[]\n    for x,is_angle in zip(q,angle_mask):\n        if is_angle: fs += [math.sin(float(x)), math.cos(float(x))]\n        else: fs += [float(x)]\n    fs.extend(map(float,v)); fs.extend(map(float,action))\n    return np.asarray(fs,dtype=float)\n\n\ndef residual_target(x,y,angle_mask):\n    n=len(angle_mask); dq=np.asarray(y[:n]-x[:n],dtype=float)\n    dq[angle_mask]=wrap_angle(dq[angle_mask]); dv=np.asarray(y[n:]-x[n:],dtype=float)\n    return np.r_[dq,dv]\n\n\nclass ResidualWM:\n    def __init__(self,input_dim,output_dim,hidden=128,layers=3):\n        import torch\n        import torch.nn as nn\n        mods=[nn.Linear(input_dim,hidden),nn.SiLU()]\n        for _ in range(layers-1): mods += [nn.Linear(hidden,hidden),nn.SiLU()]\n        mods += [nn.Linear(hidden,output_dim)]\n        self.torch=torch; self.net=nn.Sequential(*mods).double()\n        self.fmu=self.fsd=self.rmu=self.rsd=None\n\n    def fit(self,F,R,epochs,batch,lr,wd,seed):\n        torch=self.torch; seed_all(seed)\n        self.fmu=F.mean(0); self.fsd=F.std(0)+1e-8; self.rmu=R.mean(0); self.rsd=R.std(0)+1e-8\n        X=torch.tensor((F-self.fmu)/self.fsd,dtype=torch.float64)\n        Y=torch.tensor((R-self.rmu)/self.rsd,dtype=torch.float64)\n        opt=torch.optim.AdamW(self.net.parameters(),lr=lr,weight_decay=wd)\n        for _ in range(epochs):\n            order=torch.randperm(len(X))\n            for st in range(0,len(X),batch):\n                j=order[st:st+batch]; pred=self.net(X[j]); loss=((pred-Y[j])**2).mean()\n                opt.zero_grad(); loss.backward(); opt.step()\n        return self\n\n    def predict_residual(self,F):\n        torch=self.torch\n        with torch.no_grad():\n            p=self.net(torch.tensor((F-self.fmu)/self.fsd,dtype=torch.float64)).cpu().numpy()\n        return p*self.rsd+self.rmu, p\n\n    def state_dict_bundle(self):\n        return {\'state_dict\':self.net.state_dict(),\'fmu\':self.fmu,\'fsd\':self.fsd,\'rmu\':self.rmu,\'rsd\':self.rsd}\n\n\ndef collect_nominal(env_id,cfg,n,seed,split):\n    rng=np.random.default_rng(seed); A=EnvAdapter(env_id,seed)\n    states=[]; actions=[]; nexts=[]\n    mm=cfg[\'morphology_and_nuisance_ranges\']; dg=cfg[\'data_generation\']\n    qjit=max(abs(x) for x in mm[\'sensor_uniform_jitter_q_rad\']); vjit=max(abs(x) for x in mm[\'sensor_uniform_jitter_qdot\'])\n    count=0; episode=0\n    while count<n:\n        A.restore_model()\n        mass=rng.uniform(*mm[\'train_mass_multiplier\'])\n        damp=rng.uniform(*mm[\'train_nominal_damping_multiplier\'])\n        A.apply_morphology(mass,damp); A.reset(seed+episode)\n        seg_left=0; u=None\n        while count<n:\n            if seg_left<=0:\n                frac=rng.uniform(*dg[\'action_fraction_of_control_range_train\'],size=A.action_low.shape)\n                # control ranges are symmetric in frozen envs; interpolate robustly.\n                u=np.clip(frac*np.maximum(np.abs(A.action_low),np.abs(A.action_high)),A.action_low,A.action_high)\n                seg_left=int(rng.integers(1,6))\n            x=A.observed_state(jitter_q=qjit,jitter_v=vjit,rng=rng)\n            A.step(u)\n            y=A.observed_state(jitter_q=qjit,jitter_v=vjit,rng=rng)\n            states.append(x); actions.append(u.copy()); nexts.append(y); count+=1; seg_left-=1\n            if getattr(A.env,\'_elapsed_steps\',0)>=getattr(A.env.spec,\'max_episode_steps\',1000): break\n        episode+=1\n    angle_mask=A.jmap.angle_mask.copy(); A.close()\n    return np.asarray(states),np.asarray(actions),np.asarray(nexts),angle_mask\n\n\ndef make_training_arrays(X,U,Y,angle_mask):\n    F=np.vstack([feature_map(x,u,angle_mask) for x,u in zip(X,U)])\n    R=np.vstack([residual_target(x,y,angle_mask) for x,y in zip(X,Y)])\n    return F,R\n\n\ndef evaluate_model(model,X,U,Y,angle_mask):\n    F,R=make_training_arrays(X,U,Y,angle_mask); pr,pn=model.predict_residual(F)\n    n=len(angle_mask); pred=np.empty_like(Y)\n    pred[:,:n]=X[:,:n]+pr[:,:n]\n    for j,a in enumerate(angle_mask):\n        if a: pred[:,j]=wrap_angle(pred[:,j])\n    pred[:,n:]=X[:,n:]+pr[:,n:]\n    e=pred-Y\n    for j,a in enumerate(angle_mask):\n        if a: e[:,j]=wrap_angle(e[:,j])\n    persist=X-Y\n    for j,a in enumerate(angle_mask):\n        if a: persist[:,j]=wrap_angle(persist[:,j])\n    return {\n        \'aggregate_state_rmse\':float(np.sqrt(np.mean(e**2))),\n        \'persistence_rmse\':float(np.sqrt(np.mean(persist**2))),\n        \'rmse_ratio\':float(np.sqrt(np.mean(e**2))/max(1e-12,np.sqrt(np.mean(persist**2)))),\n        \'normalized_residual_target_mse\':float(np.mean((pn-(R-model.rmu)/model.rsd)**2)),\n    }\n\n\ndef fault_vector(rng,mag_range,mode=\'generic\'):\n    mag=rng.uniform(*mag_range); sign=rng.choice([-1.,1.])\n    if mode==\'threshold\': return np.array([0.,sign*mag])\n    th=rng.uniform(0,2*np.pi); return mag*np.array([math.cos(th),math.sin(th)])\n\n\ndef episode_residual_summary(model,env_id,cfg,seed,fault_class,threshold_arm=False,steps=None):\n    rng=np.random.default_rng(seed); A=EnvAdapter(env_id,seed)\n    mm=cfg[\'morphology_and_nuisance_ranges\']; fr=cfg[\'fault_ranges\']; dg=cfg[\'data_generation\']\n    A.restore_model(); A.apply_morphology(rng.uniform(*mm[\'hidden_in_hull_mass_multiplier\']),rng.uniform(*mm[\'hidden_nominal_nuisance_damping_multiplier\']))\n    encoder=np.zeros(2); actuator_bias=None; h=np.zeros(2)\n    if fault_class==\'ENCODER_OFFSET\':\n        h=fault_vector(rng,fr[\'encoder_offset_rad_abs\'],\'threshold\' if threshold_arm else \'generic\'); encoder=-h if threshold_arm else h\n    elif fault_class==\'MECHANICAL_REFERENCE_SHIFT\':\n        h=fault_vector(rng,fr[\'mechanical_reference_shift_rad_abs\'],\'threshold\' if threshold_arm else \'generic\'); A.apply_mechanical_reference(h)\n    elif fault_class==\'DYNAMICS_DAMPING_FAULT\':\n        A.apply_damping_fault(rng.uniform(*fr[\'damping_multiplier\']))\n    elif fault_class==\'ACTUATOR_BIAS\':\n        m=rng.uniform(*fr[\'actuator_bias_fraction_of_ctrl_range_abs\']); s=rng.choice([-1.,1.]); actuator_bias=np.full(A.action_low.shape,s*m*np.maximum(np.abs(A.action_low),np.abs(A.action_high)))\n    else: raise ValueError(fault_class)\n    A.reset(seed)\n    qjit=max(abs(x) for x in mm[\'sensor_uniform_jitter_q_rad\']); vjit=max(abs(x) for x in mm[\'sensor_uniform_jitter_qdot\'])\n    if steps is None: steps=cfg[\'hidden_test\'][\'trajectory_length_reacher\'] if env_id.startswith(\'Reacher\') else cfg[\'hidden_test\'][\'trajectory_length_inverted_double_pendulum\']\n    residuals=[]; energy=[]; seg_left=0; u=None\n    for t in range(steps):\n        if seg_left<=0:\n            frac=rng.uniform(*dg[\'action_fraction_of_control_range_hidden\'],size=A.action_low.shape)\n            u=np.clip(frac*np.maximum(np.abs(A.action_low),np.abs(A.action_high)),A.action_low,A.action_high); seg_left=int(rng.integers(1,6))\n        x=A.observed_state(encoder,qjit,vjit,rng); F=feature_map(x,u,A.jmap.angle_mask)[None,:]\n        pr,_=model.predict_residual(F); A.step(u,actuator_bias)\n        y=A.observed_state(encoder,qjit,vjit,rng); true_r=residual_target(x,y,A.jmap.angle_mask); r=true_r-pr[0]\n        residuals.append(r); energy.append(float(np.dot(r,r))); seg_left-=1\n    residuals=np.asarray(residuals)\n    # static theorem-aligned probes (only legitimate physical reference + sensor coordinate)\n    ucmd=np.array([0.21,-0.17]); R1=np.array([[1.,0.]]); R2=np.eye(2)\n    c1,w1=A.static_reference_probe(ucmd,encoder,R1); c2,w2=A.static_reference_probe(ucmd,encoder,R2)\n    out={\'fault_class\':fault_class,\'residual_mean\':residuals.mean(0).tolist(),\'residual_std\':residuals.std(0).tolist(),\'residual_absmax\':np.abs(residuals).max(0).tolist(),\'residual_energy_mean\':float(np.mean(energy)),\'E1_c\':c1.tolist(),\'E1_w\':w1.tolist(),\'E2_c\':c2.tolist(),\'E2_w\':w2.tolist()}\n    A.close(); return out\n\n\ndef run_dev(cfg,out,smoke=False,env_filter=None):\n    import torch\n    out.mkdir(parents=True,exist_ok=True)\n    envs=[e for e in cfg[\'environments\'] if env_filter in (None,e[\'id\'])]\n    report={\'protocol_id\':cfg[\'protocol_id\'],\'phase\':\'SMOKE\' if smoke else \'DEV\',\'hidden_unsealed\':False,\'environments\':{}}\n    model_count=2 if smoke else cfg[\'world_models\'][\'count_per_environment\']\n    for ec in envs:\n        env_id=ec[\'id\']; ntr=2500 if smoke else ec[\'train_transitions\']; ncal=1000 if smoke else ec[\'calibration_transitions\']; nte=1000 if smoke else ec[\'nominal_test_transitions\']\n        X,U,Y,am=collect_nominal(env_id,cfg,ntr,cfg[\'split_seeds\'][\'nominal_train\'], \'train\'); F,R=make_training_arrays(X,U,Y,am)\n        Xte,Ute,Yte,am2=collect_nominal(env_id,cfg,nte,cfg[\'split_seeds\'][\'calibration\']+99,\'test\')\n        if not np.array_equal(am,am2): raise RuntimeError(\'angle mask changed\')\n        envrep={\'model_quality\':[],\'dev_residual_examples\':[]}\n        for mi,seed in enumerate(cfg[\'world_models\'][\'seeds\'][:model_count]):\n            wm=ResidualWM(F.shape[1],R.shape[1],cfg[\'world_models\'][\'architecture\'].count(\'128\') and 128 or 128,3).fit(F,R,25 if smoke else cfg[\'world_models\'][\'epochs\'],cfg[\'world_models\'][\'batch_size\'],cfg[\'world_models\'][\'learning_rate\'],cfg[\'world_models\'][\'weight_decay\'],seed)\n            q=evaluate_model(wm,Xte,Ute,Yte,am); q[\'seed\']=seed; envrep[\'model_quality\'].append(q)\n            if mi==0:\n                for k,fc in enumerate(cfg[\'hidden_test\'][\'pure_fault_classes\']):\n                    envrep[\'dev_residual_examples\'].append(episode_residual_summary(wm,env_id,cfg,cfg[\'data_generation\'][\'development_seed\']+k,fc,threshold_arm=(fc in (\'ENCODER_OFFSET\',\'MECHANICAL_REFERENCE_SHIFT\')),steps=10 if smoke else None))\n            torch.save(wm.state_dict_bundle(),out/f"{env_id.replace(\'-\',\'_\')}_wm_seed{seed}.pt")\n        report[\'environments\'][env_id]=envrep\n    save_json(out/\'MEC_E2E_V1_DEV_RESULT.json\',report)\n    return report\n\n\ndef main():\n    ap=argparse.ArgumentParser()\n    ap.add_argument(\'--config\',required=True); ap.add_argument(\'--out\',required=True)\n    ap.add_argument(\'--phase\',choices=[\'smoke\',\'dev\',\'hidden\'],default=\'smoke\')\n    ap.add_argument(\'--env\',default=None)\n    ap.add_argument(\'--unseal-hidden\',default=\'NO\')\n    a=ap.parse_args(); cfg=json.loads(Path(a.config).read_text()); seed_all(12345)\n    if a.phase==\'hidden\':\n        if a.unseal_hidden!=\'YES_I_ACCEPT_FROZEN_PROTOCOL\':\n            raise SystemExit(\'HIDDEN TEST LOCKED. Use exact --unseal-hidden YES_I_ACCEPT_FROZEN_PROTOCOL only after dev engineering is frozen.\')\n        raise SystemExit(\'Hidden evaluator intentionally not implemented in the engineering runner yet. Do not unseal.\')\n    rep=run_dev(cfg,Path(a.out),smoke=(a.phase==\'smoke\'),env_filter=a.env)\n    print(json.dumps(rep,indent=2))\n\nif __name__==\'__main__\': main()\n')
(SRC/'mec_e2e_v1_dev_full.py').write_text('from __future__ import annotations\nimport argparse, json, math, hashlib, sys, time\nfrom pathlib import Path\nimport numpy as np\n\nBASE = Path(__file__).resolve().parent\nsys.path.insert(0, str(BASE))\nimport mec_e2e_v1_runner as core\n\n\ndef save_json(p, x): p.write_text(json.dumps(x, indent=2))\ndef sha256(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()\n\n\n\n\nclass FastResidualWM:\n    """Same frozen 3x128 SiLU architecture; float32 and CUDA when available (engineering-only acceleration)."""\n    def __init__(self,input_dim,output_dim,hidden=128,layers=3):\n        import torch, torch.nn as nn\n        self.torch=torch; self.device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n        mods=[nn.Linear(input_dim,hidden),nn.SiLU()]\n        for _ in range(layers-1): mods += [nn.Linear(hidden,hidden),nn.SiLU()]\n        mods += [nn.Linear(hidden,output_dim)]\n        self.net=nn.Sequential(*mods).float().to(self.device)\n        self.fmu=self.fsd=self.rmu=self.rsd=None\n    def fit(self,F,R,epochs,batch,lr,wd,seed):\n        torch=self.torch; core.seed_all(seed)\n        self.fmu=F.mean(0); self.fsd=F.std(0)+1e-8; self.rmu=R.mean(0); self.rsd=R.std(0)+1e-8\n        X=torch.tensor((F-self.fmu)/self.fsd,dtype=torch.float32)\n        Y=torch.tensor((R-self.rmu)/self.rsd,dtype=torch.float32)\n        opt=torch.optim.AdamW(self.net.parameters(),lr=lr,weight_decay=wd)\n        g=torch.Generator(device=\'cpu\'); g.manual_seed(seed)\n        for _ in range(epochs):\n            order=torch.randperm(len(X),generator=g)\n            for st in range(0,len(X),batch):\n                j=order[st:st+batch]\n                xb=X[j].to(self.device,non_blocking=True); yb=Y[j].to(self.device,non_blocking=True)\n                pred=self.net(xb); loss=((pred-yb)**2).mean(); opt.zero_grad(set_to_none=True); loss.backward(); opt.step()\n        return self\n    def predict_residual(self,F):\n        torch=self.torch\n        with torch.no_grad():\n            X=torch.tensor((F-self.fmu)/self.fsd,dtype=torch.float32,device=self.device)\n            p=self.net(X).cpu().numpy().astype(float)\n        return p*self.rsd+self.rmu,p\n    def state_dict_bundle(self):\n        return {\'state_dict\':{k:v.detach().cpu() for k,v in self.net.state_dict().items()},\n                \'fmu\':self.fmu,\'fsd\':self.fsd,\'rmu\':self.rmu,\'rsd\':self.rsd,\n                \'device_used\':str(self.device),\'dtype\':\'float32\'}\n\n\ndef residual_summary_matrix(res):\n    """Fixed episode summary; no labels or simulator params."""\n    res=np.asarray(res,float)\n    feats=[res.mean(0), res.std(0), np.mean(np.abs(res),0), np.max(np.abs(res),0),\n           np.quantile(res,0.25,axis=0), np.quantile(res,0.50,axis=0), np.quantile(res,0.75,axis=0)]\n    return np.concatenate(feats)\n\n\ndef predict_episode_residuals(model, X, U, Y, angle_mask):\n    F,R=core.make_training_arrays(np.asarray(X),np.asarray(U),np.asarray(Y),angle_mask)\n    pr,_=model.predict_residual(F)\n    return R-pr\n\n\ndef set_nonhinge_zero(A):\n    A.data.qpos[:] = A.model.qpos0\n    A.data.qvel[:] = 0\n\n\ndef common_model_setup(A, mass_mult, damping_mult):\n    A.restore_model(); A.apply_morphology(mass_mult,damping_mult)\n\n\ndef matched_static_raw_pair(env_id,cfg,seed,steps):\n    """Construct two separately evaluated systems with common h/nuisance/action stream.\n    Pairing is only dataset construction; diagnostic features never compare the two worlds.\n    """\n    rng=np.random.default_rng(seed)\n    mm=cfg[\'morphology_and_nuisance_ranges\']; fr=cfg[\'fault_ranges\']; dg=cfg[\'data_generation\']\n    mass=float(rng.uniform(*mm[\'hidden_in_hull_mass_multiplier\']))\n    damp=float(rng.uniform(*mm[\'hidden_nominal_nuisance_damping_multiplier\']))\n    mag=float(rng.uniform(*fr[\'encoder_offset_rad_abs\'])); sign=float(rng.choice([-1.,1.]))\n    h=np.array([0.,sign*mag])\n    qjit=max(abs(x) for x in mm[\'sensor_uniform_jitter_q_rad\']); vjit=max(abs(x) for x in mm[\'sensor_uniform_jitter_qdot\'])\n\n    action_fracs=[]; segs=[]; left=0; frac=None\n    # Generate command stream independent of either environment.\n    for t in range(steps):\n        if left<=0:\n            frac=rng.uniform(*dg[\'action_fraction_of_control_range_hidden\'])\n            left=int(rng.integers(1,6))\n        action_fracs.append(float(frac)); left-=1\n\n    out=[]\n    for label,kind in [(0,\'ENCODER_OFFSET\'),(1,\'MECHANICAL_REFERENCE_SHIFT\')]:\n        A=core.EnvAdapter(env_id,seed+10000+label)\n        common_model_setup(A,mass,damp)\n        enc=np.zeros(2)\n        if kind==\'ENCODER_OFFSET\': enc=-h\n        else: A.apply_mechanical_reference(h)\n        A.reset(seed+20000)\n        X=[];U=[];Y=[]\n        # Same stochastic sensor nuisance sequence for the pair.\n        srng=np.random.default_rng(seed+30000)\n        for frac in action_fracs:\n            u=np.clip(frac*np.maximum(np.abs(A.action_low),np.abs(A.action_high)),A.action_low,A.action_high)\n            x=A.observed_state(enc,qjit,vjit,srng); A.step(u); y=A.observed_state(enc,qjit,vjit,srng)\n            X.append(x);U.append(u.copy());Y.append(y)\n        ucmd=np.array([0.21,-0.17]); R1=np.array([[1.,0.]]); R2=np.eye(2)\n        c1,w1=A.static_reference_probe(ucmd,enc,R1); c2,w2=A.static_reference_probe(ucmd,enc,R2)\n        out.append({\'label\':label,\'class\':kind,\'X\':np.asarray(X),\'U\':np.asarray(U),\'Y\':np.asarray(Y),\n                    \'angle_mask\':A.jmap.angle_mask.copy(),\'E1_c\':c1,\'E1_w\':w1,\'E2_c\':c2,\'E2_w\':w2})\n        A.close()\n    return out\n\n\ndef dynamic_raw(env_id,cfg,seed,kind):\n    """Single-system diagnostic residual data at the frozen D2/D3 points."""\n    rng=np.random.default_rng(seed); mm=cfg[\'morphology_and_nuisance_ranges\']; fr=cfg[\'fault_ranges\']\n    A=core.EnvAdapter(env_id,seed+40000)\n    common_model_setup(A,float(rng.uniform(*mm[\'hidden_in_hull_mass_multiplier\'])),\n                       float(rng.uniform(*mm[\'hidden_nominal_nuisance_damping_multiplier\'])))\n    act_bias=None\n    if kind==\'DYNAMICS_DAMPING_FAULT\': A.apply_damping_fault(float(rng.uniform(*fr[\'damping_multiplier\'])))\n    elif kind==\'ACTUATOR_BIAS\':\n        m=float(rng.uniform(*fr[\'actuator_bias_fraction_of_ctrl_range_abs\'])); s=float(rng.choice([-1.,1.]))\n        act_bias=np.full(A.action_low.shape,s*m*np.maximum(np.abs(A.action_low),np.abs(A.action_high)))\n    else: raise ValueError(kind)\n\n    if env_id.startswith(\'Reacher\'):\n        spec=fr[\'dynamic_threshold_points_reacher\']; pose=np.asarray(spec[\'pose_q\']); pts=list(spec[\'velocity_points_D2\'])+[spec[\'velocity_point_added_D3\']]; ctrl=np.asarray(spec[\'control\'])\n    else:\n        spec=fr[\'dynamic_threshold_points_inverted_double_pendulum\']; pose=np.asarray(spec[\'hinge_pose_q\']); pts=list(spec[\'hinge_velocity_points_D2\'])+[spec[\'hinge_velocity_point_added_D3\']]; ctrl=np.asarray(spec[\'control\'])\n\n    rows=[]\n    for v in pts:\n        A.set_diagnostic_state(pose,np.asarray(v,float))\n        x=A.observed_state(None,0,0,None); F=core.feature_map(x,ctrl,A.jmap.angle_mask)[None,:]\n        A.step(ctrl,act_bias); y=A.observed_state(None,0,0,None)\n        R=core.residual_target(x,y,A.jmap.angle_mask); pr,_=CURRENT_MODEL.predict_residual(F)\n        rows.append(R-pr[0])\n    am=A.jmap.angle_mask.copy(); A.close()\n    return np.asarray(rows),am\n\n\ndef fit_eval_binary(X,y,groups,seed):\n    from sklearn.model_selection import GroupShuffleSplit\n    from sklearn.ensemble import RandomForestClassifier\n    from sklearn.svm import SVC\n    from sklearn.metrics import roc_auc_score,accuracy_score\n    X=np.asarray(X,float); y=np.asarray(y,int); groups=np.asarray(groups)\n    sp=GroupShuffleSplit(n_splits=1,test_size=.35,random_state=seed)\n    tr,te=next(sp.split(X,y,groups))\n    out={}\n    models={\n      \'rf\':RandomForestClassifier(n_estimators=400,min_samples_leaf=3,class_weight=\'balanced\',random_state=seed),\n      \'svm\':SVC(C=10,kernel=\'rbf\',probability=True,class_weight=\'balanced\',random_state=seed)\n    }\n    for name,m in models.items():\n        m.fit(X[tr],y[tr]); p=m.predict_proba(X[te])[:,1]; pred=(p>=.5).astype(int)\n        auc=float(roc_auc_score(y[te],p)); auc=max(auc,1-auc) # best orientation; leakage test\n        out[name]={\'auc\':auc,\'accuracy\':float(accuracy_score(y[te],pred)),\'n_train\':len(tr),\'n_test\':len(te)}\n    return out\n\n\ndef calibration_stats(model,X,U,Y,am):\n    res=predict_episode_residuals(model,X,U,Y,am)\n    energy=np.sum(res*res,axis=1)\n    return {\'residual_mean\':res.mean(0).tolist(),\'residual_std\':res.std(0).tolist(),\n            \'energy_q90\':float(np.quantile(energy,.90)),\'energy_q95\':float(np.quantile(energy,.95)),\n            \'energy_q99\':float(np.quantile(energy,.99))}\n\n\ndef train_models_for_env(env_cfg,cfg,out,smoke=False):\n    env_id=env_cfg[\'id\']; wm_cfg=cfg[\'world_models\']\n    ntr=env_cfg[\'train_transitions\']; ncal=env_cfg[\'calibration_transitions\']; nte=env_cfg[\'nominal_test_transitions\']\n    print(f\'[{env_id}] collecting nominal train/cal/test\',flush=True)\n    X,U,Y,am=core.collect_nominal(env_id,cfg,ntr,cfg[\'split_seeds\'][\'nominal_train\'],\'train\')\n    Xc,Uc,Yc,amc=core.collect_nominal(env_id,cfg,ncal,cfg[\'split_seeds\'][\'calibration\'],\'cal\')\n    Xt,Ut,Yt,amt=core.collect_nominal(env_id,cfg,nte,cfg[\'split_seeds\'][\'calibration\']+99,\'test\')\n    if not (np.array_equal(am,amc) and np.array_equal(am,amt)): raise RuntimeError(\'angle mask changed\')\n    F,R=core.make_training_arrays(X,U,Y,am)\n    models=[]; qual=[]; cal=[]\n    for i,seed in enumerate(wm_cfg[\'seeds\']):\n        print(f\'[{env_id}] train model {i+1}/8 seed={seed}\',flush=True)\n        m=FastResidualWM(F.shape[1],R.shape[1],128,3).fit(F,R,wm_cfg[\'epochs\'],wm_cfg[\'batch_size\'],wm_cfg[\'learning_rate\'],wm_cfg[\'weight_decay\'],seed)\n        q=core.evaluate_model(m,Xt,Ut,Yt,am); q[\'seed\']=seed; qual.append(q); cal.append(calibration_stats(m,Xc,Uc,Yc,am)); models.append(m)\n        try:\n            import torch; torch.save(m.state_dict_bundle(),out/f"{env_id.replace(\'-\',\'_\')}_wm_seed{seed}.pt")\n        except Exception: pass\n    return models,qual,cal\n\n\ndef dev_threshold_audits(env_id,cfg,models,seed):\n    n=cfg[\'data_generation\'][\'development_fault_episodes_per_class_per_environment\']\n    steps=cfg[\'hidden_test\'][\'trajectory_length_reacher\'] if env_id.startswith(\'Reacher\') else cfg[\'hidden_test\'][\'trajectory_length_inverted_double_pendulum\']\n    report={\'static_by_model\':[],\'dynamic_by_model\':[],\'n_pairs\':n}\n    # Generate static raw pairs once; then each model gets its own residual features.\n    print(f\'[{env_id}] generating {n} matched static development pairs\',flush=True)\n    static_pairs=[matched_static_raw_pair(env_id,cfg,seed+1000+i,steps) for i in range(n)]\n    for mi,m in enumerate(models):\n        X0=[];X1=[];X2=[];y=[];g=[]\n        for i,pair in enumerate(static_pairs):\n            for row in pair:\n                rs=predict_episode_residuals(m,row[\'X\'],row[\'U\'],row[\'Y\'],row[\'angle_mask\']); e0=residual_summary_matrix(rs)\n                X0.append(e0); X1.append(np.r_[e0,row[\'E1_c\'],row[\'E1_w\']]); X2.append(np.r_[e0,row[\'E2_c\'],row[\'E2_w\']]); y.append(row[\'label\']); g.append(i)\n        r={\'model_seed\':cfg[\'world_models\'][\'seeds\'][mi],\n           \'E0\':fit_eval_binary(X0,y,g,seed+mi),\'E1\':fit_eval_binary(X1,y,g,seed+100+mi),\'E2\':fit_eval_binary(X2,y,g,seed+200+mi)}\n        report[\'static_by_model\'].append(r)\n\n    # Dynamic raw generation depends on model because residual = learned-model error.\n    print(f\'[{env_id}] generating dynamic development audits\',flush=True)\n    global CURRENT_MODEL\n    for mi,m in enumerate(models):\n        CURRENT_MODEL=m; X2=[];X3=[];y=[];g=[]\n        for i in range(n):\n            for lab,kind in [(0,\'ACTUATOR_BIAS\'),(1,\'DYNAMICS_DAMPING_FAULT\')]:\n                rows,_=dynamic_raw(env_id,cfg,seed+50000+i,kind)\n                X2.append(rows[:2].ravel()); X3.append(rows[:3].ravel()); y.append(lab); g.append(i)\n        report[\'dynamic_by_model\'].append({\'model_seed\':cfg[\'world_models\'][\'seeds\'][mi],\n                                           \'D2\':fit_eval_binary(X2,y,g,seed+300+mi),\n                                           \'D3\':fit_eval_binary(X3,y,g,seed+400+mi)})\n    return report\n\n\ndef summarize(cfg,allrep):\n    qg=cfg[\'world_models\'][\'quality_gate\']; summary={}\n    for env_id,r in allrep.items():\n        qual=r[\'model_quality\']\n        passed=[q[\'rmse_ratio\']<=qg[\'aggregate_state_rmse_vs_persistence_ratio_max\'] and q[\'normalized_residual_target_mse\']<=qg[\'normalized_residual_target_mse_max\'] for q in qual]\n        p0=sum(passed)>=qg[\'models_required\']\n        static=r[\'development_threshold_audit\'][\'static_by_model\']; dynamic=r[\'development_threshold_audit\'][\'dynamic_by_model\']\n        best_e0=[max(x[\'E0\'][\'rf\'][\'auc\'],x[\'E0\'][\'svm\'][\'auc\']) for x in static]\n        best_e1=[max(x[\'E1\'][\'rf\'][\'auc\'],x[\'E1\'][\'svm\'][\'auc\']) for x in static]\n        best_e2=[max(x[\'E2\'][\'rf\'][\'auc\'],x[\'E2\'][\'svm\'][\'auc\']) for x in static]\n        best_d2=[max(x[\'D2\'][\'rf\'][\'auc\'],x[\'D2\'][\'svm\'][\'auc\']) for x in dynamic]\n        best_d3=[max(x[\'D3\'][\'rf\'][\'auc\'],x[\'D3\'][\'svm\'][\'auc\']) for x in dynamic]\n        summary[env_id]={\n          \'P0_dev_proxy\':p0,\'quality_models_passed\':int(sum(passed)),\n          \'static_best_auc_median\':{\'E0\':float(np.median(best_e0)),\'E1\':float(np.median(best_e1)),\'E2\':float(np.median(best_e2))},\n          \'dynamic_best_auc_median\':{\'D2\':float(np.median(best_d2)),\'D3\':float(np.median(best_d3))},\n          \'development_warning_static_below_threshold_leaks\':bool(np.median(best_e0)>.65 or np.median(best_e1)>.65),\n          \'development_warning_dynamic_D2_leaks\':bool(np.median(best_d2)>.65),\n        }\n    return summary\n\n\ndef main():\n    ap=argparse.ArgumentParser(); ap.add_argument(\'--config\',required=True); ap.add_argument(\'--out\',required=True); ap.add_argument(\'--env\',default=None)\n    a=ap.parse_args(); cfg=json.loads(Path(a.config).read_text()); out=Path(a.out); out.mkdir(parents=True,exist_ok=True)\n    if any(s in cfg[\'split_seeds\'][\'hidden_test\'] for s in [cfg[\'split_seeds\'][\'nominal_train\'],cfg[\'split_seeds\'][\'calibration\'],cfg[\'data_generation\'][\'development_seed\']]):\n        raise RuntimeError(\'development seed collides with hidden seed\')\n    report={\'protocol_id\':cfg[\'protocol_id\'],\'phase\':\'FULL_DEVELOPMENT_ONLY\',\'hidden_unsealed\':False,\n            \'config_sha256\':sha256(a.config),\'dev_runner_sha256\':sha256(__file__),\'environments\':{}}\n    envs=[e for e in cfg[\'environments\'] if a.env in (None,e[\'id\'])]\n    for ec in envs:\n        models,qual,cal=train_models_for_env(ec,cfg,out)\n        audit=dev_threshold_audits(ec[\'id\'],cfg,models,cfg[\'data_generation\'][\'development_seed\'])\n        report[\'environments\'][ec[\'id\']]={\'model_quality\':qual,\'calibration\':cal,\'development_threshold_audit\':audit}\n        save_json(out/\'MEC_E2E_V1_DEV_RESULT_PARTIAL.json\',report)\n    report[\'summary\']=summarize(cfg,report[\'environments\'])\n    save_json(out/\'MEC_E2E_V1_FULL_DEV_RESULT.json\',report)\n    print(json.dumps({\'phase\':report[\'phase\'],\'hidden_unsealed\':False,\'summary\':report[\'summary\']},indent=2))\n\nif __name__==\'__main__\': main()\n')
(SRC/'mec_e2e_v1_1_realistic_dev.py').write_text('\nfrom __future__ import annotations\nimport argparse, json, math, sys, hashlib\nfrom pathlib import Path\nimport numpy as np\n\nBASE=Path(__file__).resolve().parent\nsys.path.insert(0,str(BASE))\nimport mec_e2e_v1_runner as core\nimport mec_e2e_v1_dev_full as dev\n\nPURE = ["ENCODER_OFFSET","MECHANICAL_REFERENCE_SHIFT","DYNAMICS_DAMPING_FAULT","ACTUATOR_BIAS"]\nOOV = ["ENCODER_OFFSET+DYNAMICS_DAMPING_FAULT",\n       "MECHANICAL_REFERENCE_SHIFT+ACTUATOR_BIAS",\n       "OUTSIDE_HULL_MORPHOLOGY"]\nSTATIC_SET={"ENCODER_OFFSET","MECHANICAL_REFERENCE_SHIFT"}\nDYNAMIC_SET={"DYNAMICS_DAMPING_FAULT","ACTUATOR_BIAS"}\n\ndef save_json(p,x): Path(p).write_text(json.dumps(x,indent=2))\ndef sha256(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()\n\ndef residual_summary(res):\n    res=np.asarray(res,float)\n    return np.concatenate([\n        res.mean(0), res.std(0), np.mean(np.abs(res),0),\n        np.max(np.abs(res),0),\n        np.quantile(res,.25,axis=0), np.quantile(res,.50,axis=0),\n        np.quantile(res,.75,axis=0)\n    ])\n\ndef random_vec2(rng, mag_range):\n    mag=float(rng.uniform(*mag_range)); th=float(rng.uniform(0,2*np.pi))\n    return mag*np.array([np.cos(th),np.sin(th)])\n\ndef action_stream(rng,A,steps,frac_range):\n    out=[]; left=0; frac=None\n    mx=np.maximum(np.abs(A.action_low),np.abs(A.action_high))\n    for _ in range(steps):\n        if left<=0:\n            frac=rng.uniform(*frac_range,size=A.action_low.shape)\n            left=int(rng.integers(1,6))\n        out.append(np.clip(frac*mx,A.action_low,A.action_high)); left-=1\n    return out\n\ndef apply_fault(A, old, rng, kind, allow_oov=False):\n    mm=old["morphology_and_nuisance_ranges"]; fr=old["fault_ranges"]\n    encoder=np.zeros(2); actuator=np.zeros_like(A.action_low)\n    # in-hull nuisance first\n    mass=float(rng.uniform(*mm["hidden_in_hull_mass_multiplier"]))\n    damp=float(rng.uniform(*mm["hidden_nominal_nuisance_damping_multiplier"]))\n    if kind=="OUTSIDE_HULL_MORPHOLOGY":\n        lo = rng.random()<.5\n        mass=float(rng.uniform(*(mm["hidden_oov_mass_multiplier_low"] if lo else mm["hidden_oov_mass_multiplier_high"])))\n    A.restore_model(); A.apply_morphology(mass,damp)\n\n    def add_encoder():\n        nonlocal encoder\n        encoder += random_vec2(rng,fr["encoder_offset_rad_abs"])\n    def add_mech():\n        A.apply_mechanical_reference(random_vec2(rng,fr["mechanical_reference_shift_rad_abs"]))\n    def add_damp():\n        A.apply_damping_fault(float(rng.uniform(*fr["damping_multiplier"])))\n    def add_act():\n        nonlocal actuator\n        # target second actuator only; physically closer to dynamic intervention pair.\n        frac=float(rng.uniform(*fr["actuator_bias_fraction_of_ctrl_range_abs"]))\n        s=float(rng.choice([-1.,1.])); mx=np.maximum(np.abs(A.action_low),np.abs(A.action_high))\n        actuator[1 if len(actuator)>1 else 0] += s*frac*mx[1 if len(actuator)>1 else 0]\n\n    if kind=="ENCODER_OFFSET": add_encoder()\n    elif kind=="MECHANICAL_REFERENCE_SHIFT": add_mech()\n    elif kind=="DYNAMICS_DAMPING_FAULT": add_damp()\n    elif kind=="ACTUATOR_BIAS": add_act()\n    elif kind=="ENCODER_OFFSET+DYNAMICS_DAMPING_FAULT": add_encoder(); add_damp()\n    elif kind=="MECHANICAL_REFERENCE_SHIFT+ACTUATOR_BIAS": add_mech(); add_act()\n    elif kind=="OUTSIDE_HULL_MORPHOLOGY": pass\n    else: raise ValueError(kind)\n    return encoder, actuator, mass, damp\n\ndef make_scenario(old, seed, kind, steps=40):\n    rng=np.random.default_rng(seed)\n    A=core.EnvAdapter("Reacher-v5",seed+10000)\n    enc,act,mass,damp=apply_fault(A,old,rng,kind)\n    A.reset(seed+20000)\n    mm=old["morphology_and_nuisance_ranges"]; dg=old["data_generation"]\n    qjit=max(abs(x) for x in mm["sensor_uniform_jitter_q_rad"])\n    vjit=max(abs(x) for x in mm["sensor_uniform_jitter_qdot"])\n    acts=action_stream(rng,A,steps,dg["action_fraction_of_control_range_hidden"])\n    srng=np.random.default_rng(seed+30000)\n    X=[];U=[];Y=[]\n    for u in acts:\n        x=A.observed_state(enc,qjit,vjit,srng)\n        A.step(u,act if np.any(act) else None)\n        y=A.observed_state(enc,qjit,vjit,srng)\n        X.append(x);U.append(u.copy());Y.append(y)\n    # Full-rank physical reference probe.\n    ucmd=np.array([0.21,-0.17]); c2,w2=A.static_reference_probe(ucmd,enc,np.eye(2))\n    # Three diagnostic transitions from actual single system.\n    spec=old["fault_ranges"]["dynamic_threshold_points_reacher"]\n    pose=np.asarray(spec["pose_q"],float)\n    pts=[np.array([0.0,0.20]),np.array([0.55,0.20]),np.array([0.0,0.75])]\n    D=[]\n    for v in pts:\n        A.set_diagnostic_state(pose,v)\n        x=A.observed_state(enc,0,0,None)\n        u=np.zeros_like(A.action_low)\n        A.step(u,act if np.any(act) else None)\n        y=A.observed_state(enc,0,0,None)\n        D.append((x,u.copy(),y))\n    am=A.jmap.angle_mask.copy(); A.close()\n    return {\n      "kind":kind,"X":np.asarray(X),"U":np.asarray(U),"Y":np.asarray(Y),\n      "angle_mask":am,"static":np.r_[c2,w2],\n      "dyn_X":np.asarray([z[0] for z in D]),"dyn_U":np.asarray([z[1] for z in D]),\n      "dyn_Y":np.asarray([z[2] for z in D]),\n      "meta":{"mass":mass,"nominal_nuisance_damping":damp}\n    }\n\ndef scenario_features(model,row):\n    F,R=core.make_training_arrays(row["X"],row["U"],row["Y"],row["angle_mask"])\n    pr,_=model.predict_residual(F); passive=residual_summary(R-pr)\n    Fd,Rd=core.make_training_arrays(row["dyn_X"],row["dyn_U"],row["dyn_Y"],row["angle_mask"])\n    pd,_=model.predict_residual(Fd); dyn=(Rd-pd).ravel()\n    static=np.asarray(row["static"],float)\n    return {"passive":passive,\n            "static":np.r_[passive,static],\n            "dynamic":np.r_[passive,dyn],\n            "full":np.r_[passive,static,dyn],\n            "static_probe":static,"dynamic_probe":dyn}\n\ndef split_groups(n,seed=32117):\n    rng=np.random.default_rng(seed); ids=np.arange(n); rng.shuffle(ids)\n    ntr=n//2; ncal=n//4\n    return set(ids[:ntr]),set(ids[ntr:ntr+ncal]),set(ids[ntr+ncal:])\n\ndef fit_stage_models(X_by_stage,y,is_oov,train_mask,seed):\n    from sklearn.ensemble import RandomForestClassifier\n    from sklearn.pipeline import make_pipeline\n    from sklearn.preprocessing import StandardScaler\n    from sklearn.neural_network import MLPClassifier\n    out={}\n    pure=(~is_oov)&train_mask\n    anytr=train_mask\n    for j,stage in enumerate(["passive","static","dynamic","full"]):\n        X=np.asarray(X_by_stage[stage],float)\n        rf=RandomForestClassifier(n_estimators=500,min_samples_leaf=3,class_weight="balanced",random_state=seed+j)\n        rf.fit(X[pure],y[pure])\n        oov=RandomForestClassifier(n_estimators=400,min_samples_leaf=3,class_weight="balanced",random_state=seed+100+j)\n        oov.fit(X[anytr],is_oov[anytr].astype(int))\n        out[stage]={"class_rf":rf,"oov_rf":oov}\n    X=np.asarray(X_by_stage["passive"],float)\n    mlp=make_pipeline(StandardScaler(),\n        MLPClassifier(hidden_layer_sizes=(128,64),alpha=1e-3,max_iter=700,early_stopping=True,\n                      validation_fraction=.2,random_state=seed+700))\n    mlp.fit(X[pure],y[pure])\n    out["passive_mlp"]=mlp\n    return out\n\ndef choose_class_threshold(probs,y,mask,target=.05):\n    p=np.asarray(probs); yy=np.asarray(y)\n    conf=p.max(1); pred=p.argmax(1)\n    vals=np.unique(np.r_[0.0,conf[mask],1.000001])\n    best=(1.000001,0.0,0.0)\n    for t in vals:\n        take=mask&(conf>=t); n=take.sum()\n        if n==0: continue\n        risk=float(np.mean(pred[take]!=yy[take])); cov=float(n/max(1,mask.sum()))\n        if risk<=target+1e-12 and cov>best[1]: best=(float(t),cov,risk)\n    return {"threshold":best[0],"coverage":best[1],"risk":best[2]}\n\ndef choose_oov_threshold(p_oov,is_oov,mask,target_reject=.80):\n    p=np.asarray(p_oov,float); oo=np.asarray(is_oov,bool)\n    vals=np.unique(np.r_[0.0,p[mask],1.000001])\n    best=None\n    for t in vals:\n        reject=p>=t\n        om=mask&oo; pm=mask&(~oo)\n        orate=float(np.mean(reject[om])) if om.any() else 0.0\n        false=float(np.mean(reject[pm])) if pm.any() else 1.0\n        if orate>=target_reject:\n            cand=(false,-orate,float(t),orate)\n            if best is None or cand<best: best=cand\n    if best is None:\n        # highest achievable reject with lowest pure false reject\n        arr=[]\n        for t in vals:\n            reject=p>=t; om=mask&oo; pm=mask&(~oo)\n            orate=float(np.mean(reject[om])) if om.any() else 0.0\n            false=float(np.mean(reject[pm])) if pm.any() else 1.0\n            arr.append((-orate,false,float(t),orate))\n        q=min(arr); return {"threshold":q[2],"oov_reject":q[3],"pure_false_reject":q[1],"target_met":False}\n    return {"threshold":best[2],"oov_reject":best[3],"pure_false_reject":best[0],"target_met":True}\n\ndef standardized_pair_separation(X,y,mask,class_a,class_b):\n    xx=np.asarray(X,float); yy=np.asarray(y)\n    z=mask&np.isin(yy,[class_a,class_b])\n    xa=xx[z&(yy==class_a)]; xb=xx[z&(yy==class_b)]\n    if len(xa)<2 or len(xb)<2: return 0.0\n    mu=xa.mean(0)-xb.mean(0)\n    sd=np.sqrt(.5*(xa.var(0)+xb.var(0)))+1e-6\n    return float(np.sqrt(np.mean((mu/sd)**2)))\n\ndef calibrate(models,X_by_stage,y,is_oov,cal_mask,target_risk=.05,target_oov=.80):\n    cal={"class":{},"oov":{}}\n    for stage in ["passive","static","dynamic","full"]:\n        X=np.asarray(X_by_stage[stage],float)\n        probs=models[stage]["class_rf"].predict_proba(X)\n        # classes are 0..3\n        pure=cal_mask&(~is_oov)\n        cal["class"][stage]=choose_class_threshold(probs,y,pure,target_risk)\n        po=models[stage]["oov_rf"].predict_proba(X)[:,1]\n        cal["oov"][stage]=choose_oov_threshold(po,is_oov,cal_mask,target_oov)\n    return cal\n\ndef class_probs(models,stage,x):\n    return models[stage]["class_rf"].predict_proba(np.asarray(x,float)[None,:])[0]\n\ndef oov_prob(models,stage,x):\n    return float(models[stage]["oov_rf"].predict_proba(np.asarray(x,float)[None,:])[0,1])\n\ndef stage_decision(models,cal,stage,x):\n    po=oov_prob(models,stage,x)\n    if po>=cal["oov"][stage]["threshold"]:\n        return {"status":"ESCALATE","pred":None,"conf":None}\n    p=class_probs(models,stage,x); k=int(np.argmax(p)); conf=float(np.max(p))\n    if conf>=cal["class"][stage]["threshold"]:\n        return {"status":"COMMIT","pred":k,"conf":conf}\n    return {"status":"UNCERTAIN","pred":k,"conf":conf,"probs":p}\n\ndef probe_choice_mec(p):\n    top=np.argsort(p)[-2:]; s=set(top.tolist())\n    if s=={0,1}: return "static"\n    if s=={2,3}: return "dynamic"\n    # Mixed semantic ambiguity: pick probe aimed at the top class family.\n    return "static" if int(np.argmax(p)) in (0,1) else "dynamic"\n\ndef build_sep_table(Xs,Xd,y,train_mask):\n    tab={}\n    for a in range(4):\n        for b in range(a+1,4):\n            tab[(a,b)]={\n              "static":standardized_pair_separation(Xs,y,train_mask,a,b),\n              "dynamic":standardized_pair_separation(Xd,y,train_mask,a,b)}\n    return tab\n\ndef probe_choice_active(p,sep):\n    top=sorted(np.argsort(p)[-2:].tolist()); q=sep[tuple(top)]\n    return "static" if q["static"]>=q["dynamic"] else "dynamic"\n\nCOST={"passive":0,"static":2,"dynamic":3,"full":5}\n\ndef adaptive_policy(models,cal,feat,mode,sep=None,rng=None):\n    d=stage_decision(models,cal,"passive",feat["passive"])\n    if d["status"]!="UNCERTAIN":\n        return d["status"],d.get("pred"),0\n    p=d["probs"]\n    if mode=="MEC": first=probe_choice_mec(p)\n    elif mode=="ACTIVE": first=probe_choice_active(p,sep)\n    elif mode=="RANDOM": first=("static" if rng.random()<.5 else "dynamic")\n    else: raise ValueError(mode)\n    d1=stage_decision(models,cal,first,feat[first])\n    if d1["status"]!="UNCERTAIN":\n        return d1["status"],d1.get("pred"),COST[first]\n    # acquire the other evidence channel and use full evidence.\n    d2=stage_decision(models,cal,"full",feat["full"])\n    return d2["status"] if d2["status"]!="UNCERTAIN" else "ABSTAIN", d2.get("pred"), COST["full"]\n\ndef metrics_policy(outcomes,true_y,is_oov,costs):\n    oo=np.asarray(is_oov,bool); y=np.asarray(true_y)\n    committed=np.array([s=="COMMIT" for s,_ in outcomes])\n    pred=np.array([-1 if p is None else p for _,p in outcomes])\n    pure=~oo\n    cp=committed&pure\n    wrong=float(np.mean(pred[cp]!=y[cp])) if cp.any() else 0.0\n    cov=float(np.mean(committed[pure])) if pure.any() else 0.0\n    correct=float(np.mean((pred==y)&committed&pure)) if pure.any() else 0.0\n    oov_wrong=float(np.mean(committed[oo])) if oo.any() else 0.0\n    oov_safe=float(np.mean(~committed[oo])) if oo.any() else 0.0\n    return {"wrong_intervention_risk":wrong,"coverage":cov,"correct_intervention_rate":correct,\n            "combined_oov_wrong_pure_intervention":oov_wrong,\n            "combined_oov_abstain_or_escalate":oov_safe,\n            "mean_probe_cost":float(np.mean(costs[pure])) if pure.any() else 0.0}\n\ndef passive_metrics(model,cal,X,y,is_oov,stage="passive",mlp=False):\n    outcomes=[]; costs=[]\n    for i,x in enumerate(X):\n        po=oov_prob(model,stage,x)\n        if po>=cal["oov"][stage]["threshold"]:\n            outcomes.append(("ESCALATE",None)); costs.append(0); continue\n        if mlp:\n            p=model["passive_mlp"].predict_proba(np.asarray(x)[None,:])[0]\n            # use RF-calibrated passive threshold to avoid post-hoc extra threshold family\n            t=cal["class"]["passive"]["threshold"]\n        else:\n            p=class_probs(model,stage,x); t=cal["class"][stage]["threshold"]\n        if float(np.max(p))>=t: outcomes.append(("COMMIT",int(np.argmax(p))))\n        else: outcomes.append(("ABSTAIN",None))\n        costs.append(0)\n    return metrics_policy(outcomes,y,is_oov,np.asarray(costs))\n\ndef evaluate_one_model(model,rows,new,old,model_seed):\n    labels={k:i for i,k in enumerate(PURE)}\n    n=max(r["group"] for r in rows)+1\n    tr,ca,te=split_groups(n,32117)\n    group=np.array([r["group"] for r in rows])\n    train=np.array([g in tr for g in group]); calmask=np.array([g in ca for g in group]); test=np.array([g in te for g in group])\n    is_oov=np.array([r["kind"] in OOV for r in rows])\n    y=np.array([labels.get(r["kind"],-1) for r in rows])\n    feats=[scenario_features(model,r["raw"]) for r in rows]\n    X={s:np.vstack([f[s] for f in feats]) for s in ["passive","static","dynamic","full"]}\n    stage=fit_stage_models(X,y,is_oov,train,model_seed+2000)\n    cal=calibrate(stage,X,y,is_oov,calmask,.05,.80)\n    sep=build_sep_table(X["static"],X["dynamic"],y,train&(~is_oov))\n    idx=np.where(test)[0]\n    rr=np.random.default_rng(model_seed+91000)\n    results={}\n    for mode in ["MEC","RANDOM","ACTIVE"]:\n        outs=[]; costs=[]\n        for i in idx:\n            s,p,c=adaptive_policy(stage,cal,feats[i],mode,sep,rr)\n            outs.append((s,p)); costs.append(c)\n        results[mode]=metrics_policy(outs,y[idx],is_oov[idx],np.asarray(costs))\n    results["PASSIVE_RF"]=passive_metrics(stage,cal,X["passive"][idx],y[idx],is_oov[idx],"passive",False)\n    results["PASSIVE_MLP"]=passive_metrics(stage,cal,X["passive"][idx],y[idx],is_oov[idx],"passive",True)\n    # Full-evidence upper bound, same selective logic but no probe-cost penalty.\n    results["FULL_EVIDENCE_ORACLE"]=passive_metrics(stage,cal,X["full"][idx],y[idx],is_oov[idx],"full",False)\n    ratio=results["MEC"]["mean_probe_cost"]/max(1e-12,results["RANDOM"]["mean_probe_cost"])\n    gates=new["realistic_arm"]["gates"]\n    proxy={\n      "wrong_risk":results["MEC"]["wrong_intervention_risk"]<=gates["wrong_intervention_risk_max"],\n      "coverage":results["MEC"]["coverage"]>=gates["coverage_min"],\n      "oov_wrong":results["MEC"]["combined_oov_wrong_pure_intervention"]<=gates["combined_oov_wrong_pure_intervention_max"],\n      "oov_safe":results["MEC"]["combined_oov_abstain_or_escalate"]>=gates["combined_oov_abstain_or_escalate_min"],\n      "random_cost_ratio":ratio<=gates["random_probe_cost_ratio_max"]\n    }\n    return {"model_seed":model_seed,"calibration":cal,\n            "probe_pair_separation":{f"{a}-{b}":v for (a,b),v in sep.items()},\n            "metrics":results,"mec_to_random_cost_ratio":ratio,"development_gate_proxy":proxy,\n            "n_train":int(train.sum()),"n_cal":int(calmask.sum()),"n_test":int(test.sum())}\n\ndef generate_rows(old,n=120,seed0=733001):\n    rows=[]\n    # group-aligned generation: every group contains all pure + all OOV types.\n    for g in range(n):\n        for j,kind in enumerate(PURE+OOV):\n            raw=make_scenario(old,seed0 + g*100 + j,kind,40)\n            rows.append({"group":g,"kind":kind,"raw":raw})\n    return rows\n\ndef quality(model,Xt,Ut,Yt,am):\n    return core.evaluate_model(model,Xt,Ut,Yt,am)\n\ndef main():\n    ap=argparse.ArgumentParser()\n    ap.add_argument("--old-config",required=True); ap.add_argument("--new-config",required=True)\n    ap.add_argument("--out",required=True); ap.add_argument("--groups",type=int,default=120)\n    ap.add_argument("--models",type=int,default=8)\n    a=ap.parse_args()\n    old=json.loads(Path(a.old_config).read_text()); new=json.loads(Path(a.new_config).read_text())\n    out=Path(a.out); out.mkdir(parents=True,exist_ok=True)\n\n    forbidden=set(new["hidden_seeds"])|set(new["world_models"]["hidden_seeds"])\n    used=set(new["world_models"]["development_seeds"][:a.models])|{\n        old["split_seeds"]["nominal_train"],old["split_seeds"]["calibration"],733001}\n    if forbidden & used: raise RuntimeError("development seed collides with v1.1 hidden seeds")\n\n    # Train full-development models.\n    print("collect nominal train/cal/test",flush=True)\n    X,U,Y,am=core.collect_nominal("Reacher-v5",old,new["world_models"]["train_transitions"],\n                                  old["split_seeds"]["nominal_train"],"train")\n    Xt,Ut,Yt,amt=core.collect_nominal("Reacher-v5",old,new["world_models"]["test_transitions"],\n                                      old["split_seeds"]["calibration"]+99,"test")\n    F,R=core.make_training_arrays(X,U,Y,am)\n    models=[]; q=[]\n    for i,seed in enumerate(new["world_models"]["development_seeds"][:a.models]):\n        print(f"train development model {i+1}/{a.models} seed={seed}",flush=True)\n        m=dev.FastResidualWM(F.shape[1],R.shape[1],128,3).fit(\n            F,R,new["world_models"]["epochs"],old["world_models"]["batch_size"],\n            old["world_models"]["learning_rate"],old["world_models"]["weight_decay"],seed)\n        qq=quality(m,Xt,Ut,Yt,am); qq["seed"]=seed; q.append(qq); models.append(m)\n\n    print(f"generate {a.groups} group-aligned realistic scenarios x {len(PURE)+len(OOV)} classes",flush=True)\n    rows=generate_rows(old,a.groups,733001)\n    by=[]\n    for i,m in enumerate(models):\n        print(f"evaluate realistic arm model {i+1}/{len(models)}",flush=True)\n        by.append(evaluate_one_model(m,rows,new,old,new["world_models"]["development_seeds"][i]))\n\n    qgate=new["world_models"]["quality_gate"]\n    qpass=[z["rmse_ratio"]<=qgate["rmse_ratio_max"] and\n           z["normalized_residual_target_mse"]<=qgate["normalized_residual_target_mse_max"] for z in q]\n    # Aggregate medians; development proxy only, not hidden result.\n    modes=["MEC","RANDOM","ACTIVE","PASSIVE_RF","PASSIVE_MLP","FULL_EVIDENCE_ORACLE"]\n    agg={}\n    for mode in modes:\n        keys=by[0]["metrics"][mode].keys()\n        agg[mode]={k:float(np.median([z["metrics"][mode][k] for z in by])) for k in keys}\n    costratio=float(np.median([z["mec_to_random_cost_ratio"] for z in by]))\n    gate_names=by[0]["development_gate_proxy"].keys()\n    gate_fraction={k:float(np.mean([z["development_gate_proxy"][k] for z in by])) for k in gate_names}\n    summary={\n      "hidden_unsealed":False,\n      "quality_models_passed":int(sum(qpass)),\n      "P0_dev_proxy":sum(qpass)>=qgate["models_required"],\n      "median_metrics":agg,\n      "median_mec_to_random_cost_ratio":costratio,\n      "fraction_models_passing_each_realistic_proxy_gate":gate_fraction,\n      "development_ready_for_policy_freeze": bool(\n          sum(qpass)>=qgate["models_required"] and\n          gate_fraction["wrong_risk"]>=.75 and gate_fraction["coverage"]>=.75 and\n          gate_fraction["oov_wrong"]>=.75 and gate_fraction["oov_safe"]>=.75)\n    }\n    result={"protocol_id":new["protocol_id"],"phase":"V1_1_REALISTIC_DEVELOPMENT_ONLY",\n            "hidden_unsealed":False,"old_config_sha256":sha256(a.old_config),\n            "new_config_sha256":sha256(a.new_config),\n            "quality":q,"by_model":by,"summary":summary}\n    save_json(out/"MEC_E2E_V1_1_REALISTIC_DEV_RESULT.json",result)\n    print(json.dumps(summary,indent=2))\n\nif __name__=="__main__": main()\n')
(SRC/'mec_e2e_v1_3_compositional_voi_dev.py').write_text('\nfrom __future__ import annotations\nimport argparse, json, sys, hashlib\nfrom pathlib import Path\nimport numpy as np\n\nBASE=Path(__file__).resolve().parent\nsys.path.insert(0,str(BASE))\nimport mec_e2e_v1_runner as core\nimport mec_e2e_v1_dev_full as dev\nimport mec_e2e_v1_1_realistic_dev as v11\n\nPURE=v11.PURE\nOOV=v11.OOV\nALL=PURE+OOV\n\nBITS={\n "ENCODER_OFFSET":np.array([1,0,0,0],int),\n "MECHANICAL_REFERENCE_SHIFT":np.array([0,1,0,0],int),\n "DYNAMICS_DAMPING_FAULT":np.array([0,0,1,0],int),\n "ACTUATOR_BIAS":np.array([0,0,0,1],int),\n "ENCODER_OFFSET+DYNAMICS_DAMPING_FAULT":np.array([1,0,1,0],int),\n "MECHANICAL_REFERENCE_SHIFT+ACTUATOR_BIAS":np.array([0,1,0,1],int),\n "OUTSIDE_HULL_MORPHOLOGY":np.array([0,0,0,0],int),\n}\n\nCOST={"static":2.0,"dynamic":3.0,"full":5.0}\n\ndef save_json(p,x): Path(p).write_text(json.dumps(x,indent=2))\ndef sha256(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()\n\ndef nominal_norm(model,X,U,Y,am):\n    F,R=core.make_training_arrays(X,U,Y,am)\n    pr,_=model.predict_residual(F)\n    rr=R-pr\n    return rr.mean(0),rr.std(0)+1e-6\n\ndef summary(rr):\n    rr=np.asarray(rr,float)\n    return np.concatenate([rr.mean(0),rr.std(0),np.mean(np.abs(rr),0),np.max(np.abs(rr),0),\n                           np.quantile(rr,.25,axis=0),np.quantile(rr,.5,axis=0),np.quantile(rr,.75,axis=0)])\n\ndef feat(model,row,mu,sd):\n    F,R=core.make_training_arrays(row["X"],row["U"],row["Y"],row["angle_mask"])\n    pr,_=model.predict_residual(F)\n    passive=summary((R-pr-mu)/sd)\n    Fd,Rd=core.make_training_arrays(row["dyn_X"],row["dyn_U"],row["dyn_Y"],row["angle_mask"])\n    pd,_=model.predict_residual(Fd)\n    dyn=((Rd-pd-mu)/sd).ravel()\n    static=np.asarray(row["static"],float)\n    return {"passive":passive,"static":np.r_[passive,static],\n            "dynamic":np.r_[passive,dyn],"full":np.r_[passive,static,dyn]}\n\ndef split_groups(n,seed):\n    rng=np.random.default_rng(seed); ids=np.arange(n); rng.shuffle(ids)\n    return set(ids[:60]),set(ids[60:90]),set(ids[90:120])\n\ndef build_records(models,norms,rows,seeds):\n    rec=[]\n    for m,(mu,sd),seed in zip(models,norms,seeds):\n        for r in rows:\n            rec.append({"wm":seed,"group":r["group"],"kind":r["kind"],\n                        "bits":BITS[r["kind"]],"outside":int(r["kind"]=="OUTSIDE_HULL_MORPHOLOGY"),\n                        "f":feat(m,r["raw"],mu,sd)})\n    return rec\n\ndef arrays(rec):\n    X={s:np.vstack([r["f"][s] for r in rec]) for s in ["passive","static","dynamic","full"]}\n    B=np.vstack([r["bits"] for r in rec]).astype(int)\n    O=np.array([r["outside"] for r in rec],int)\n    W=np.array([r["wm"] for r in rec]); G=np.array([r["group"] for r in rec])\n    return X,B,O,W,G\n\ndef fit_heads(X,B,O,mask,seed):\n    from sklearn.ensemble import RandomForestClassifier\n    out={}\n    for j,s in enumerate(["passive","static","dynamic","full"]):\n        xx=np.asarray(X[s],float); heads=[]\n        for k in range(4):\n            c=RandomForestClassifier(n_estimators=600,min_samples_leaf=3,class_weight="balanced",\n                                     max_features="sqrt",n_jobs=-1,random_state=seed+20*j+k)\n            c.fit(xx[mask],B[mask,k]); heads.append(c)\n        od=RandomForestClassifier(n_estimators=600,min_samples_leaf=3,class_weight="balanced",\n                                  max_features="sqrt",n_jobs=-1,random_state=seed+200+j)\n        od.fit(xx[mask],O[mask])\n        out[s]={"heads":heads,"outside":od}\n    return out\n\ndef head_probs(models,s,X):\n    xx=np.asarray(X,float)\n    ps=[]\n    for h in models[s]["heads"]:\n        p=h.predict_proba(xx)\n        if p.shape[1]==1:\n            val=np.full(len(xx),float(h.classes_[0]==1))\n        else:\n            idx=list(h.classes_).index(1)\n            val=p[:,idx]\n        ps.append(val)\n    return np.vstack(ps).T\n\ndef out_prob(models,s,X):\n    h=models[s]["outside"]; p=h.predict_proba(np.asarray(X,float))\n    if p.shape[1]==1: return np.full(len(X),float(h.classes_[0]==1))\n    return p[:,list(h.classes_).index(1)]\n\ndef bin_entropy(p):\n    p=np.clip(np.asarray(p,float),1e-6,1-1e-6)\n    return -(p*np.log(p)+(1-p)*np.log(1-p)).sum(axis=1)\n\ndef fit_voi(models,X,train,seed):\n    from sklearn.ensemble import RandomForestRegressor\n    pp=head_probs(models,"passive",X["passive"])\n    ps=head_probs(models,"static",X["static"])\n    pd=head_probs(models,"dynamic",X["dynamic"])\n    base=bin_entropy(pp)\n    ys=np.maximum(0.0,base-bin_entropy(ps))/COST["static"]\n    yd=np.maximum(0.0,base-bin_entropy(pd))/COST["dynamic"]\n    rs=RandomForestRegressor(n_estimators=500,min_samples_leaf=4,max_features="sqrt",n_jobs=-1,random_state=seed)\n    rd=RandomForestRegressor(n_estimators=500,min_samples_leaf=4,max_features="sqrt",n_jobs=-1,random_state=seed+1)\n    rs.fit(X["passive"][train],ys[train]); rd.fit(X["passive"][train],yd[train])\n    return rs,rd\n\ndef calibrate(models,X,B,O,cal):\n    # Grid-search a common mechanism-positive threshold and "other" ceiling.\n    # Optimize pure coverage subject to <=5% wrong pure interventions and <=5% unsafe pure commits on non-pure cases.\n    P={s:head_probs(models,s,X[s]) for s in ["static","dynamic","full"]}\n    OP={s:out_prob(models,s,X[s]) for s in ["static","dynamic","full"]}\n    out={}\n    pure=(B.sum(1)==1)\n    nonpure=~pure\n    for s in ["static","dynamic","full"]:\n        best=None\n        for tp in np.linspace(.50,.95,19):\n            for to in np.linspace(.05,.45,17):\n                for oo in np.linspace(.35,.90,12):\n                    pp=P[s]; op=OP[s]\n                    top=pp.argmax(1); topv=pp.max(1)\n                    sec=np.sort(pp,axis=1)[:,-2]\n                    commit=(topv>=tp)&(sec<=to)&(op<oo)\n                    pm=cal&pure; nm=cal&nonpure\n                    wrong=np.mean(top[pm&commit] != B[pm&commit].argmax(1)) if np.any(pm&commit) else 0.0\n                    cov=np.mean(commit[pm]) if np.any(pm) else 0.0\n                    unsafe=np.mean(commit[nm]) if np.any(nm) else 0.0\n                    if wrong<=.05+1e-12 and unsafe<=.05+1e-12:\n                        # maximize coverage, then minimize unsafe, then lower threshold burden.\n                        cand=(cov,-unsafe,-wrong,-tp,to,-oo)\n                        if best is None or cand>best[0]:\n                            best=(cand,{"positive":float(tp),"other_max":float(to),"outside_max":float(oo),\n                                       "cal_wrong":float(wrong),"cal_coverage":float(cov),"cal_nonpure_commit":float(unsafe)})\n        if best is None:\n            out[s]={"positive":1.01,"other_max":0.0,"outside_max":0.0,\n                    "cal_wrong":0.0,"cal_coverage":0.0,"cal_nonpure_commit":0.0}\n        else:\n            out[s]=best[1]\n    return out\n\ndef stage_decision(models,thr,s,x,final=False):\n    p=head_probs(models,s,np.asarray(x)[None,:])[0]\n    op=float(out_prob(models,s,np.asarray(x)[None,:])[0])\n    t=thr[s]\n    supported=np.where(p>=t["positive"])[0]\n    top=int(np.argmax(p)); sec=float(np.sort(p)[-2])\n    # Explicit composite detection dominates pure commit.\n    if len(supported)>=2:\n        return "ESCALATE",None,p,op\n    if op>=t["outside_max"]:\n        return ("ESCALATE" if final else "MORE"),None,p,op\n    if float(p[top])>=t["positive"] and sec<=t["other_max"]:\n        return "COMMIT",top,p,op\n    return ("ABSTAIN" if final else "MORE"),None,p,op\n\ndef discrim_probe(passive_p):\n    # Semantic ambiguity heuristic retained only as comparator.\n    top=np.argsort(passive_p)[-2:]\n    static=sum(int(k in (0,1)) for k in top)\n    dynamic=sum(int(k in (2,3)) for k in top)\n    return "static" if static>=dynamic else "dynamic"\n\ndef policy(models,thr,voi,features,mode,rng):\n    pp=head_probs(models,"passive",np.asarray(features["passive"])[None,:])[0]\n    if mode=="MEC_VOI":\n        us=float(voi[0].predict(np.asarray(features["passive"])[None,:])[0])\n        ud=float(voi[1].predict(np.asarray(features["passive"])[None,:])[0])\n        first="static" if us>=ud else "dynamic"\n    elif mode=="RANDOM":\n        first="static" if rng.random()<.5 else "dynamic"\n    elif mode=="DISCRIM":\n        first=discrim_probe(pp)\n    else:\n        raise ValueError(mode)\n    st,p,_,_=stage_decision(models,thr,first,features[first],False)\n    if st=="COMMIT": return st,p,COST[first]\n    sf,pf,_,_=stage_decision(models,thr,"full",features["full"],True)\n    return sf,pf,COST["full"]\n\ndef score(outs,B,cost):\n    B=np.asarray(B,int); pure=B.sum(1)==1; nonpure=~pure\n    commit=np.array([a=="COMMIT" for a,_ in outs])\n    pred=np.array([-1 if b is None else b for _,b in outs])\n    true=np.argmax(B,axis=1)\n    cp=commit&pure\n    wrong=np.mean(pred[cp]!=true[cp]) if np.any(cp) else 0.0\n    cov=np.mean(commit[pure]) if np.any(pure) else 0.0\n    correct=np.sum((pred[pure]==true[pure])&commit[pure])/max(1,pure.sum())\n    unsafe=np.mean(commit[nonpure]) if np.any(nonpure) else 0.0\n    return {"wrong_intervention_risk":float(wrong),"coverage":float(cov),\n            "correct_intervention_rate":float(correct),"composite_or_ood_pure_commit":float(unsafe),\n            "composite_or_ood_safe":float(1-unsafe),\n            "mean_probe_cost":float(np.mean(np.asarray(cost)[pure])) if np.any(pure) else 0.0}\n\ndef fold_eval(rec,hold,trg,cag,teg,seed):\n    X,B,O,W,G=arrays(rec)\n    train=(W!=hold)&np.isin(G,list(trg))\n    cal=(W!=hold)&np.isin(G,list(cag))\n    test=(W==hold)&np.isin(G,list(teg))\n    models=fit_heads(X,B,O,train,seed)\n    voi=fit_voi(models,X,train,seed+500)\n    thr=calibrate(models,X,B,O,cal)\n    idx=np.where(test)[0]; rng=np.random.default_rng(seed+900)\n    res={}\n    for mode,label in [("MEC_VOI","MEC_VOI"),("RANDOM","RANDOM"),("DISCRIM","DISCRIMINATION_HEURISTIC")]:\n        outs=[]; costs=[]\n        for i in idx:\n            a,b,c=policy(models,thr,voi,{s:X[s][i] for s in X},mode,rng)\n            outs.append((a,b)); costs.append(c)\n        res[label]=score(outs,B[idx],costs)\n    ratio=res["MEC_VOI"]["mean_probe_cost"]/max(1e-12,res["RANDOM"]["mean_probe_cost"])\n    m=res["MEC_VOI"]\n    primary=(m["wrong_intervention_risk"]<=.05 and m["coverage"]>=.70 and\n             m["composite_or_ood_pure_commit"]<=.05 and m["composite_or_ood_safe"]>=.95)\n    eff=ratio<=.85\n    return {"heldout_world_model_seed":int(hold),"n_train":int(train.sum()),"n_cal":int(cal.sum()),"n_test":int(test.sum()),\n            "thresholds":thr,"metrics":res,"mec_to_random_probe_cost_ratio":float(ratio),\n            "primary_pass":bool(primary),"efficiency_pass":bool(eff)}\n\ndef main():\n    ap=argparse.ArgumentParser()\n    ap.add_argument("--old-config",required=True)\n    ap.add_argument("--v13-config",required=True)\n    ap.add_argument("--out",required=True)\n    a=ap.parse_args()\n    old=json.loads(Path(a.old_config).read_text()); cfg=json.loads(Path(a.v13_config).read_text())\n    out=Path(a.out); out.mkdir(parents=True,exist_ok=True)\n    forbidden=set(cfg["hidden_world_model_seeds_forbidden"])|set(cfg["hidden_scenario_seeds_forbidden"])\n    used=set(cfg["development_world_model_seeds"])|{cfg["scenario_design"]["development_seed0"],\n          cfg["scenario_design"]["group_split_seed"],old["split_seeds"]["nominal_train"],old["split_seeds"]["calibration"]}\n    if forbidden&used: raise RuntimeError("hidden seed collision")\n\n    print("collect nominal datasets",flush=True)\n    X,U,Y,am=core.collect_nominal("Reacher-v5",old,60000,old["split_seeds"]["nominal_train"],"train")\n    Xc,Uc,Yc,amc=core.collect_nominal("Reacher-v5",old,20000,old["split_seeds"]["calibration"],"cal")\n    Xt,Ut,Yt,amt=core.collect_nominal("Reacher-v5",old,20000,old["split_seeds"]["calibration"]+99,"test")\n    F,R=core.make_training_arrays(X,U,Y,am)\n\n    models=[]; norms=[]; quality=[]\n    seeds=cfg["development_world_model_seeds"]\n    for j,seed in enumerate(seeds):\n        print(f"train development WM {j+1}/8 seed={seed}",flush=True)\n        m=dev.FastResidualWM(F.shape[1],R.shape[1],128,3).fit(\n            F,R,120,old["world_models"]["batch_size"],old["world_models"]["learning_rate"],\n            old["world_models"]["weight_decay"],seed)\n        q=core.evaluate_model(m,Xt,Ut,Yt,am); q["seed"]=seed; quality.append(q)\n        norms.append(nominal_norm(m,Xc,Uc,Yc,am)); models.append(m)\n\n    print("generate v1.3 development scenarios",flush=True)\n    rows=v11.generate_rows(old,cfg["scenario_design"]["groups"],cfg["scenario_design"]["development_seed0"])\n    rec=build_records(models,norms,rows,seeds)\n    tr,ca,te=split_groups(cfg["scenario_design"]["groups"],cfg["scenario_design"]["group_split_seed"])\n\n    folds=[]\n    for j,hold in enumerate(seeds):\n        print(f"LOMO fold {j+1}/8 hold={hold}",flush=True)\n        folds.append(fold_eval(rec,hold,tr,ca,te,93000+j))\n\n    qpass=[q["rmse_ratio"]<=.5 and q["normalized_residual_target_mse"]<=.25 for q in quality]\n    primary_n=sum(f["primary_pass"] for f in folds)\n    eff_n=sum(f["efficiency_pass"] for f in folds)\n    keys=folds[0]["metrics"]["MEC_VOI"].keys()\n    med={k:float(np.median([f["metrics"]["MEC_VOI"][k] for f in folds])) for k in keys}\n    ratio=float(np.median([f["mec_to_random_probe_cost_ratio"] for f in folds]))\n    median_primary=(med["wrong_intervention_risk"]<=.05 and med["coverage"]>=.70 and\n                    med["composite_or_ood_pure_commit"]<=.05 and med["composite_or_ood_safe"]>=.95)\n    primary_pass=(sum(qpass)>=7 and primary_n>=6 and median_primary)\n    efficiency_pass=(ratio<=.85 and eff_n>=6)\n    if primary_pass and efficiency_pass: decision="PRIMARY_PASS_EFFICIENCY_PASS"\n    elif primary_pass: decision="PRIMARY_PASS_EFFICIENCY_FAIL"\n    else: decision="PRIMARY_FAIL"\n    summary={"hidden_unsealed":False,"quality_models_passed":int(sum(qpass)),\n             "primary_folds_passed":int(primary_n),"efficiency_folds_passed":int(eff_n),\n             "median_MEC_VOI":med,"median_mec_to_random_probe_cost_ratio":ratio,\n             "median_primary_pass":bool(median_primary),"decision":decision}\n    res={"protocol_id":cfg["protocol_id"],"phase":"V1_3_COMPOSITIONAL_VOI_DEVELOPMENT",\n         "hidden_unsealed":False,"quality":quality,"folds":folds,"summary":summary}\n    save_json(out/"MEC_E2E_V1_3_COMPOSITIONAL_VOI_DEV_RESULT.json",res)\n    print(json.dumps(summary,indent=2))\n\nif __name__=="__main__":\n    main()\n')
(SRC/'mec_e2e_v1_4_robust_set_commit_dev.py').write_text('from __future__ import annotations\nimport argparse, json, os, pickle, sys, time, hashlib\nfrom pathlib import Path\nimport numpy as np\n\nBASE=Path(__file__).resolve().parent\nsys.path.insert(0,str(BASE))\nimport mec_e2e_v1_runner as core\nimport mec_e2e_v1_dev_full as dev\nimport mec_e2e_v1_1_realistic_dev as v11\nimport mec_e2e_v1_3_compositional_voi_dev as v13\n\nPURE=v11.PURE\nOOV=v11.OOV\nALL=PURE+OOV\nCOST={"static":2.0,"dynamic":3.0,"full":5.0}\nSTAGES=["passive","static","dynamic","full"]\nPATCH_ID="MEC_V1_4_R1_ROBUST_SET_CERTIFICATE_FAST_RESUMABLE"\n\nSTATE_NAMES=[\n "ENCODER_OFFSET","MECHANICAL_REFERENCE_SHIFT","DYNAMICS_DAMPING_FAULT","ACTUATOR_BIAS",\n "ENCODER_OFFSET+DYNAMICS_DAMPING_FAULT","MECHANICAL_REFERENCE_SHIFT+ACTUATOR_BIAS","OUTSIDE_HULL_MORPHOLOGY"]\nSTATE_ID={k:i for i,k in enumerate(STATE_NAMES)}\n\ndef atomic_json(path,obj):\n    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True); tmp=Path(str(path)+".tmp")\n    tmp.write_text(json.dumps(obj,indent=2)); os.replace(tmp,path)\n\ndef atomic_pickle(path,obj):\n    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True); tmp=Path(str(path)+".tmp")\n    with open(tmp,"wb") as f: pickle.dump(obj,f,pickle.HIGHEST_PROTOCOL); f.flush(); os.fsync(f.fileno())\n    os.replace(tmp,path)\n\ndef load_pickle(path):\n    with open(path,"rb") as f: return pickle.load(f)\n\ndef progress(ckpt,stage,**kw):\n    x={"stage":stage,"updated_unix":time.time(),**kw}; atomic_json(Path(ckpt)/"PROGRESS_V14.json",x)\n    print("\\n"+"="*68+"\\nPROGRESS "+json.dumps(x,indent=2)+"\\n"+"="*68,flush=True)\n\ndef sha256(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()\n\ndef split_old_train(n,seed):\n    rng=np.random.default_rng(seed); ids=np.arange(n); rng.shuffle(ids); return set(ids[:60])\n\ndef split_fresh(n,seed):\n    rng=np.random.default_rng(seed); ids=np.arange(n); rng.shuffle(ids); return set(ids[:60]),set(ids[60:120])\n\ndef infer_model_dims(obj):\n    sd=obj["bundle"]["state_dict"]\n    ws=[v for k,v in sd.items() if k.endswith("weight")]\n    return int(ws[0].shape[1]), int(ws[-1].shape[0])\n\ndef load_r2_model(r2,seed):\n    import torch\n    p=Path(r2)/f"world_model_{seed}.pt"\n    if not p.exists(): raise FileNotFoundError(f"Missing R2 world model: {p}")\n    obj=torch.load(p,map_location="cpu",weights_only=False)\n    inp,out=infer_model_dims(obj)\n    m=dev.FastResidualWM(inp,out,128,3); m.net.load_state_dict(obj["bundle"]["state_dict"])\n    m.fmu=np.asarray(obj["bundle"]["fmu"]); m.fsd=np.asarray(obj["bundle"]["fsd"])\n    m.rmu=np.asarray(obj["bundle"]["rmu"]); m.rsd=np.asarray(obj["bundle"]["rsd"])\n    return m,obj["quality"],(np.asarray(obj["nominal_mu"]),np.asarray(obj["nominal_sd"]))\n\ndef get_fresh_scenarios(old,cfg,path):\n    n=int(cfg["fresh_v14_qualification"]["groups"]); seed0=int(cfg["fresh_v14_qualification"]["scenario_seed0"])\n    if Path(path).exists():\n        st=load_pickle(path); rows=st["rows"]; start=int(st["next_group"])\n        if start>=n: print("[resume] fresh scenarios complete",flush=True); return rows\n    else: rows=[]; start=0\n    for g in range(start,n):\n        for j,kind in enumerate(ALL):\n            raw=v11.make_scenario(old,seed0+g*100+j,kind,int(cfg["fresh_v14_qualification"]["steps"]))\n            rows.append({"group":g,"kind":kind,"raw":raw})\n        if (g+1)%5==0 or g+1==n:\n            atomic_pickle(path,{"next_group":g+1,"rows":rows}); print(f"[fresh scenarios] {g+1}/{n} groups [saved]",flush=True)\n    return rows\n\ndef records_to_arrays(rec):\n    X={s:np.vstack([r["f"][s] for r in rec]) for s in STAGES}\n    B=np.vstack([r["bits"] for r in rec]).astype(int)\n    W=np.asarray([r["wm"] for r in rec]); G=np.asarray([r["group"] for r in rec])\n    K=np.asarray([STATE_ID[r["kind"]] for r in rec],int)\n    return X,B,W,G,K\n\ndef fit_binary_hgb(x,y,seed,cfg):\n    from sklearn.ensemble import HistGradientBoostingClassifier\n    h=HistGradientBoostingClassifier(max_iter=int(cfg["head"]["max_iter"]),learning_rate=float(cfg["head"]["learning_rate"]),\n       max_leaf_nodes=int(cfg["head"]["max_leaf_nodes"]),l2_regularization=float(cfg["head"]["l2_regularization"]),\n       class_weight="balanced",random_state=int(seed),early_stopping=True,validation_fraction=.15,n_iter_no_change=12)\n    h.fit(np.asarray(x,float),np.asarray(y,int)); return h\n\ndef fit_state_hgb(x,y,seed,cfg):\n    from sklearn.ensemble import HistGradientBoostingClassifier\n    h=HistGradientBoostingClassifier(max_iter=int(cfg["head"]["max_iter"]),learning_rate=float(cfg["head"]["learning_rate"]),\n       max_leaf_nodes=int(cfg["head"]["max_leaf_nodes"]),l2_regularization=float(cfg["head"]["l2_regularization"]),\n       class_weight="balanced",random_state=int(seed),early_stopping=True,validation_fraction=.15,n_iter_no_change=12)\n    h.fit(np.asarray(x,float),np.asarray(y,int)); return h\n\ndef fit_heads(X,B,K,mask,seed,cfg):\n    out={}\n    for j,s in enumerate(STAGES):\n        bits=[fit_binary_hgb(X[s][mask],B[mask,k],seed+100*j+k,cfg) for k in range(4)]\n        state=fit_state_hgb(X[s][mask],K[mask],seed+500+j,cfg)\n        out[s]={"bits":bits,"state":state}\n    return out\n\ndef bit_probs(models,s,x):\n    xx=np.asarray(x,float); ps=[]\n    for h in models[s]["bits"]:\n        p=h.predict_proba(xx); classes=list(h.classes_)\n        ps.append(p[:,classes.index(1)] if 1 in classes else np.zeros(len(xx)))\n    return np.vstack(ps).T\n\ndef state_probs(models,s,x):\n    h=models[s]["state"]; p=h.predict_proba(np.asarray(x,float)); out=np.zeros((len(p),7),float)\n    for j,c in enumerate(h.classes_): out[:,int(c)]=p[:,j]\n    return out\n\ndef bin_entropy(p):\n    p=np.clip(np.asarray(p,float),1e-6,1-1e-6); return -(p*np.log(p)+(1-p)*np.log(1-p)).sum(1)\n\ndef fit_voi(models,X,train,seed):\n    from sklearn.ensemble import HistGradientBoostingRegressor\n    pp=bit_probs(models,"passive",X["passive"]); ps=bit_probs(models,"static",X["static"]); pd=bit_probs(models,"dynamic",X["dynamic"])\n    base=bin_entropy(pp); ys=np.maximum(0,base-bin_entropy(ps))/COST["static"]; yd=np.maximum(0,base-bin_entropy(pd))/COST["dynamic"]\n    rs=HistGradientBoostingRegressor(max_iter=50,learning_rate=.08,max_leaf_nodes=15,l2_regularization=.1,random_state=seed,early_stopping=True)\n    rd=HistGradientBoostingRegressor(max_iter=50,learning_rate=.08,max_leaf_nodes=15,l2_regularization=.1,random_state=seed+1,early_stopping=True)\n    rs.fit(X["passive"][train],ys[train]); rd.fit(X["passive"][train],yd[train]); return rs,rd\n\ndef certificate_arrays(models,s,x):\n    bp=bit_probs(models,s,x); sp=state_probs(models,s,x)\n    top=bp.argmax(1); ordered=np.sort(bp,axis=1); sec=ordered[:,-2]\n    state_pure=sp[np.arange(len(sp)),top]; nonpure=sp[:,4:].sum(1)\n    score=np.minimum.reduce([bp[np.arange(len(bp)),top],1-sec,state_pure,1-nonpure])\n    return score,top,bp,sp\n\ndef cp_upper(errors,n,conf=.95):\n    from scipy.stats import beta\n    errors=int(errors); n=int(n)\n    if n<=0: return 1.0\n    if errors>=n: return 1.0\n    return float(beta.ppf(conf,errors+1,n-errors))\n\ndef block_stats(score,pred,B,K,W,mask,tau):\n    pure=(B.sum(1)==1); nonpure=~pure; commit=(score>=tau)&mask\n    cp=commit&pure; npmask=mask&nonpure\n    wrong=int(np.sum(pred[cp]!=np.argmax(B[cp],axis=1))) if np.any(cp) else 0\n    ncommit=int(cp.sum()); unsafe=int((commit&nonpure).sum()); nnon=int(npmask.sum())\n    cov=float((commit&pure).sum()/max(1,(mask&pure).sum()))\n    return {"wrong":wrong,"committed_pure":ncommit,"wrong_ucb":cp_upper(wrong,ncommit),\n            "unsafe":unsafe,"nonpure_n":nnon,"unsafe_ucb":cp_upper(unsafe,nnon),"coverage":cov}\n\ndef calibrate_stage(models,s,X,B,K,W,cal,source_wms):\n    score,pred,_,_=certificate_arrays(models,s,X[s])\n    vals=np.unique(score[cal]);\n    if len(vals)>250:\n        vals=np.unique(np.quantile(vals,np.linspace(0,1,250)))\n    candidates=np.unique(np.r_[0.0,vals,1.000001])\n    best=None\n    for tau in candidates:\n        pooled=block_stats(score,pred,B,K,W,cal,float(tau))\n        blocks=[]; safe=(pooled["wrong_ucb"]<=.05 and pooled["unsafe_ucb"]<=.05)\n        for wm in source_wms:\n            b=block_stats(score,pred,B,K,W,cal&(W==wm),float(tau)); blocks.append((int(wm),b))\n            safe=safe and b["wrong_ucb"]<=.05 and b["unsafe_ucb"]<=.05\n        if not safe: continue\n        mincov=min(b["coverage"] for _,b in blocks); cand=(mincov,pooled["coverage"],-float(tau))\n        if best is None or cand>best[0]: best=(cand,float(tau),pooled,blocks)\n    if best is None:\n        return {"tau":1.000001,"calibration_found":False,"pooled":{},"per_source_wm":[]}\n    return {"tau":best[1],"calibration_found":True,"pooled":best[2],"per_source_wm":[{"wm":w,**b} for w,b in best[3]]}\n\ndef calibrate(models,X,B,K,W,cal,source_wms):\n    return {s:calibrate_stage(models,s,X,B,K,W,cal,source_wms) for s in ["static","dynamic","full"]}\n\ndef stage_decision(models,thr,s,x,final=False):\n    score,pred,_,_=certificate_arrays(models,s,np.asarray(x)[None,:]); ok=float(score[0])>=float(thr[s]["tau"])\n    if ok: return "COMMIT",int(pred[0]),float(score[0])\n    return ("ESCALATE" if final else "MORE"),None,float(score[0])\n\ndef discrim_probe(passive_p):\n    top=np.argsort(passive_p)[-2:]; st=sum(int(k in (0,1)) for k in top); dy=sum(int(k in (2,3)) for k in top); return "static" if st>=dy else "dynamic"\n\ndef policy(models,thr,voi,features,mode,rng):\n    pp=bit_probs(models,"passive",np.asarray(features["passive"])[None,:])[0]\n    if mode=="MEC_VOI":\n        us=float(voi[0].predict(np.asarray(features["passive"])[None,:])[0]); ud=float(voi[1].predict(np.asarray(features["passive"])[None,:])[0]); first="static" if us>=ud else "dynamic"\n    elif mode=="RANDOM": first="static" if rng.random()<.5 else "dynamic"\n    elif mode=="DISCRIM": first=discrim_probe(pp)\n    else: raise ValueError(mode)\n    a,b,_=stage_decision(models,thr,first,features[first],False)\n    if a=="COMMIT": return a,b,COST[first]\n    a,b,_=stage_decision(models,thr,"full",features["full"],True); return a,b,COST["full"]\n\ndef score_outputs(outs,B,cost):\n    B=np.asarray(B,int); pure=B.sum(1)==1; nonpure=~pure\n    commit=np.array([a=="COMMIT" for a,_ in outs]); pred=np.array([-1 if b is None else b for _,b in outs]); true=np.argmax(B,axis=1)\n    cp=commit&pure; wrong=np.mean(pred[cp]!=true[cp]) if np.any(cp) else 0.0; cov=np.mean(commit[pure]) if np.any(pure) else 0.0\n    correct=np.sum((pred[pure]==true[pure])&commit[pure])/max(1,pure.sum()); unsafe=np.mean(commit[nonpure]) if np.any(nonpure) else 0.0\n    return {"wrong_intervention_risk":float(wrong),"coverage":float(cov),"correct_intervention_rate":float(correct),\n            "composite_or_ood_pure_commit":float(unsafe),"composite_or_ood_safe":float(1-unsafe),\n            "mean_probe_cost":float(np.mean(np.asarray(cost)[pure])) if np.any(pure) else 0.0}\n\ndef fold_eval(oldrec,newrec,hold,old_train_groups,new_cal_groups,new_test_groups,seed,cfg):\n    Xo,Bo,Wo,Go,Ko=records_to_arrays(oldrec); Xn,Bn,Wn,Gn,Kn=records_to_arrays(newrec)\n    train=(Wo!=hold)&np.isin(Go,list(old_train_groups)); cal=(Wn!=hold)&np.isin(Gn,list(new_cal_groups)); test=(Wn==hold)&np.isin(Gn,list(new_test_groups))\n    models=fit_heads(Xo,Bo,Ko,train,seed,cfg); voi=fit_voi(models,Xo,train,seed+700)\n    src=[w for w in cfg["development_world_model_seeds"] if w!=hold]; thr=calibrate(models,Xn,Bn,Kn,Wn,cal,src)\n    idx=np.where(test)[0]; rng=np.random.default_rng(seed+900); res={}\n    for mode,label in [("MEC_VOI","MEC_VOI"),("RANDOM","RANDOM"),("DISCRIM","DISCRIMINATION_HEURISTIC")]:\n        outs=[]; costs=[]\n        for i in idx:\n            a,b,c=policy(models,thr,voi,{s:Xn[s][i] for s in STAGES},mode,rng); outs.append((a,b)); costs.append(c)\n        res[label]=score_outputs(outs,Bn[idx],costs)\n    ratio=res["MEC_VOI"]["mean_probe_cost"]/max(1e-12,res["RANDOM"]["mean_probe_cost"]); m=res["MEC_VOI"]\n    primary=(m["wrong_intervention_risk"]<=.05 and m["coverage"]>=.70 and m["composite_or_ood_pure_commit"]<=.05 and m["composite_or_ood_safe"]>=.95)\n    return {"heldout_world_model_seed":int(hold),"n_old_train":int(train.sum()),"n_fresh_cal":int(cal.sum()),"n_fresh_test":int(test.sum()),\n            "thresholds":thr,"metrics":res,"mec_to_random_probe_cost_ratio":float(ratio),"primary_pass":bool(primary),"efficiency_pass":bool(ratio<=.85)}\n\ndef main():\n    ap=argparse.ArgumentParser(); ap.add_argument("--old-config",required=True); ap.add_argument("--v14-config",required=True)\n    ap.add_argument("--r2-checkpoint-dir",required=True); ap.add_argument("--checkpoint-dir",required=True); ap.add_argument("--result-dir",required=True)\n    a=ap.parse_args(); old=json.loads(Path(a.old_config).read_text()); cfg=json.loads(Path(a.v14_config).read_text())\n    r2=Path(a.r2_checkpoint_dir); ck=Path(a.checkpoint_dir); out=Path(a.result_dir); ck.mkdir(parents=True,exist_ok=True); out.mkdir(parents=True,exist_ok=True)\n    forbidden=set(cfg["hidden_world_model_seeds_forbidden"])|set(cfg["hidden_scenario_seeds_forbidden"]); used=set(cfg["development_world_model_seeds"])|{cfg["fresh_v14_qualification"]["scenario_seed0"],cfg["fresh_v14_qualification"]["split_seed"]}\n    if forbidden&used: raise RuntimeError("HIDDEN SEED COLLISION")\n    print(f"\\n{PATCH_ID}\\nHIDDEN SEEDS REMAIN SEALED\\nWORLD MODELS ARE REUSED, NOT RETRAINED\\n",flush=True)\n    manifest={"protocol_id":cfg["protocol_id"],"v14_config_sha256":sha256(a.v14_config),"hidden_unsealed":False,"r2_source":str(r2)}\n    mp=ck/"V14_R1_MANIFEST.json"\n    if mp.exists() and json.loads(mp.read_text()).get("v14_config_sha256")!=manifest["v14_config_sha256"]: raise RuntimeError("v1.4 config changed; use new checkpoint folder")\n    if not mp.exists(): atomic_json(mp,manifest)\n\n    # Load frozen R2 world models/norms and old v1.3 feature records.\n    models=[]; norms=[]; oldrec=[]; quality=[]\n    for seed in cfg["development_world_model_seeds"]:\n        progress(ck,"LOAD_FROZEN_R2",seed=int(seed)); m,q,n=load_r2_model(r2,seed); models.append(m); quality.append(q); norms.append(n)\n        rp=r2/f"records_world_model_{seed}_R2.pkl"\n        if not rp.exists(): raise FileNotFoundError(f"Missing old v1.3 feature cache {rp}")\n        oldrec.extend(load_pickle(rp)); print(f"[loaded] WM + old records {seed}",flush=True)\n\n    progress(ck,"FRESH_SCENARIOS")\n    rows=get_fresh_scenarios(old,cfg,ck/"fresh_v14_scenarios.pkl")\n\n    newrec=[]\n    for j,(m,norm,seed) in enumerate(zip(models,norms,cfg["development_world_model_seeds"])):\n        progress(ck,"FRESH_FEATURE_RECORDS",index=j+1,total=8,seed=int(seed)); rp=ck/f"fresh_records_{seed}.pkl"\n        if rp.exists(): rr=load_pickle(rp); print(f"[resume] fresh records {seed}",flush=True)\n        else:\n            rr=v13.build_records([m],[norm],rows,[seed]); atomic_pickle(rp,rr); print(f"[saved] fresh records {seed}",flush=True)\n        newrec.extend(rr)\n\n    oldtrain=split_old_train(120,int(cfg["old_v13_training"]["group_split_seed"])); calg,testg=split_fresh(120,int(cfg["fresh_v14_qualification"]["split_seed"]))\n    folds=[]\n    for j,hold in enumerate(cfg["development_world_model_seeds"]):\n        progress(ck,"LOMO_V14",index=j+1,total=8,heldout_world_model=int(hold)); fp=ck/f"fold_v14_{hold}.json"\n        if fp.exists(): f=json.loads(fp.read_text()); print(f"[resume] v1.4 fold {hold}",flush=True)\n        else:\n            t=time.time(); f=fold_eval(oldrec,newrec,hold,oldtrain,calg,testg,104000+j,cfg); atomic_json(fp,f)\n            m=f["metrics"]["MEC_VOI"]; print(f\'[saved] fold {hold} in {(time.time()-t)/60:.2f} min | primary={f["primary_pass"]} eff={f["efficiency_pass"]} | wrong={m["wrong_intervention_risk"]:.3f} cov={m["coverage"]:.3f} unsafe={m["composite_or_ood_pure_commit"]:.3f}\',flush=True)\n        folds.append(f)\n\n    qpass=[q["rmse_ratio"]<=.5 and q["normalized_residual_target_mse"]<=.25 for q in quality]; pn=sum(f["primary_pass"] for f in folds); en=sum(f["efficiency_pass"] for f in folds)\n    keys=folds[0]["metrics"]["MEC_VOI"].keys(); med={k:float(np.median([f["metrics"]["MEC_VOI"][k] for f in folds])) for k in keys}; ratio=float(np.median([f["mec_to_random_probe_cost_ratio"] for f in folds]))\n    mpas=(med["wrong_intervention_risk"]<=.05 and med["coverage"]>=.70 and med["composite_or_ood_pure_commit"]<=.05 and med["composite_or_ood_safe"]>=.95)\n    primary=(sum(qpass)>=7 and pn>=6 and mpas); eff=(ratio<=.85 and en>=6)\n    decision="PRIMARY_PASS_EFFICIENCY_PASS" if primary and eff else ("PRIMARY_PASS_EFFICIENCY_FAIL" if primary else "PRIMARY_FAIL")\n    summary={"hidden_unsealed":False,"quality_models_passed":int(sum(qpass)),"primary_folds_passed":int(pn),"efficiency_folds_passed":int(en),"median_MEC_VOI":med,"median_mec_to_random_probe_cost_ratio":ratio,"median_primary_pass":bool(mpas),"decision":decision}\n    result={"protocol_id":cfg["protocol_id"],"implementation":PATCH_ID,"hidden_unsealed":False,"folds":folds,"summary":summary}\n    atomic_json(out/"MEC_E2E_V1_4_R1_FINAL_RESULT.json",result); atomic_json(ck/"FINAL_RESULT_V14_R1.json",result); progress(ck,"COMPLETE",decision=decision)\n    print("\\nFINAL V1.4 SUMMARY\\n"+json.dumps(summary,indent=2),flush=True)\n\nif __name__=="__main__": main()\n')
(SRC/'MEC_V1_4_FINAL_POLICY_FREEZE_CONFIG.json').write_text('{\n  "protocol_id": "MEC_V1_4_FINAL_POLICY_FREEZE_2026_08_30",\n  "status": "FROZEN_BEFORE_FINAL_POLICY_FIT",\n  "hidden_status": "SEALED",\n  "authorized_by_development_decision": "PRIMARY_PASS_EFFICIENCY_PASS",\n  "development_policy_protocol_id": "MEC_E2E_V1_4_ROBUST_SET_COMMIT_DEV_2026_08_29",\n  "source_checkpoint_dirs": {\n    "r2": "/content/drive/MyDrive/MEC_V1_3_R2_CHECKPOINTS",\n    "v14": "/content/drive/MyDrive/MEC_V1_4_R1_CHECKPOINTS"\n  },\n  "output_dir": "/content/drive/MyDrive/MEC_V1_4_FINAL_FREEZE",\n  "final_fit": {\n    "world_models": [\n      681101,\n      681102,\n      681103,\n      681104,\n      681105,\n      681106,\n      681107,\n      681108\n    ],\n    "head_fit_data": "all 8 development WMs, old v1.3 train groups only",\n    "voi_fit_data": "all 8 development WMs, old v1.3 train groups only",\n    "threshold_calibration_data": "all 8 development WMs, fresh v1.4 calibration groups only",\n    "v14_qualification_test_groups_used": false,\n    "hidden_data_used": false,\n    "old_train_group_split_seed": 43117,\n    "fresh_calibration_split_seed": 54217,\n    "fresh_calibration_groups": 60\n  },\n  "immutable_method": {\n    "head": "ROBUST_SET_CERTIFICATE_HGB",\n    "certificate": "min(top_bit_probability, 1-second_bit_probability, matching_pure_state_probability, 1-nonpure_state_probability)",\n    "calibration": "one-sided 95% Clopper-Pearson <=5% wrong-pure and <=5% nonpure pure-commit for every source WM and pooled",\n    "probe_policy": "MEC_VOI from v1.4",\n    "costs": {\n      "static": 2.0,\n      "dynamic": 3.0,\n      "full": 5.0\n    }\n  },\n  "prohibited": [\n    "v1.4 qualification-test labels in final fit",\n    "hidden seeds/data",\n    "world-model retraining",\n    "threshold relaxation",\n    "policy modification after final freeze"\n  ]\n}')
(SRC/'MEC_V1_4_REACHER_HIDDEN_FINAL_FROZEN_CONFIG.json').write_text('{\n  "protocol_id": "MEC_V1_4_REACHER_HIDDEN_FINAL_2026_08_30",\n  "status": "FROZEN_BEFORE_HIDDEN_UNSEAL",\n  "policy_freeze_protocol_id": "MEC_V1_4_FINAL_POLICY_FREEZE_2026_08_30",\n  "environment": "Reacher-v5",\n  "hidden_world_model_seeds": [\n    781101,\n    781102,\n    781103,\n    781104,\n    781105,\n    781106,\n    781107,\n    781108\n  ],\n  "hidden_scenario_seed_roots": [\n    957101,\n    957107,\n    957109\n  ],\n  "hidden_groups_per_seed_root": 200,\n  "mechanism_classes_per_group": 7,\n  "steps": 40,\n  "scenario_seed_derivation": "uint32 SeedSequence([root, group, class_index])",\n  "balanced_per_class_per_root": true,\n  "policy": "load exact serialized final policy bundle; no fitting/calibration on hidden records",\n  "world_model_training": {\n    "nominal_data": "reuse frozen R2 nominal_train/calibration/test datasets",\n    "architecture": "Residual MLP 3x128 SiLU float32",\n    "epochs": 120,\n    "batch_size": 512,\n    "learning_rate": 0.002,\n    "weight_decay": 1e-05,\n    "checkpoint_every_epochs": 5\n  },\n  "primary_hidden_gates": {\n    "quality_models_required": 7,\n    "wrong_intervention_risk_max": 0.05,\n    "pure_coverage_min": 0.7,\n    "composite_or_ood_pure_commit_max": 0.05,\n    "composite_or_ood_safe_min": 0.95,\n    "hidden_world_model_primary_passes_required": 6,\n    "of_hidden_world_models": 8,\n    "median_hidden_wm_metrics_must_pass": true,\n    "pooled_metrics_must_pass": true,\n    "seed_stability_max_wms_wrong_risk_above_0_07": 1\n  },\n  "secondary_efficiency": {\n    "median_mec_to_random_cost_ratio_max": 0.85,\n    "hidden_wm_efficiency_passes_required": 6,\n    "efficiency_failure_does_not_override_primary": true\n  },\n  "scenario_seed_aggregates": "all three roots reported; pooled/WM gates control qualification",\n  "baselines": [\n    "MEC_VOI",\n    "RANDOM",\n    "DISCRIMINATION_HEURISTIC"\n  ],\n  "unseal_token": "YES_I_ACCEPT_FINAL_FROZEN_POLICY_AND_ONE_SHOT_HIDDEN",\n  "outcome_labels": {\n    "HIDDEN_PRIMARY_PASS_EFFICIENCY_PASS": "Reacher hidden final passes; proceed to statistics/reproduction/paper freeze.",\n    "HIDDEN_PRIMARY_PASS_EFFICIENCY_FAIL": "Reacher hidden primary passes; efficiency superiority claim removed.",\n    "HIDDEN_PRIMARY_FAIL": "Hidden final fails. Hidden seeds are permanently spent and cannot be reused for tuning."\n  },\n  "prohibited": [\n    "hidden threshold fitting",\n    "hidden head/VOI fitting",\n    "changing policy bundle",\n    "dropping hidden WMs/seeds after unseal",\n    "post-hoc gate relaxation",\n    "reusing these hidden seeds as hidden after inspection"\n  ]\n}')
(SRC/'MEC_PRE_HIDDEN_AUTHORITY_MANIFEST.json').write_text('{\n  "status": "FROZEN_BEFORE_FINAL_POLICY_FIT_AND_HIDDEN_UNSEAL",\n  "created_date": "2026-08-30",\n  "hidden_unsealed": false,\n  "development_disposition": {\n    "hidden_unsealed": false,\n    "quality_models_passed": 8,\n    "primary_folds_passed": 7,\n    "efficiency_folds_passed": 7,\n    "median_MEC_VOI": {\n      "wrong_intervention_risk": 0.0,\n      "coverage": 0.7645833333333334,\n      "correct_intervention_rate": 0.7645833333333334,\n      "composite_or_ood_pure_commit": 0.005555555555555556,\n      "composite_or_ood_safe": 0.9944444444444445,\n      "mean_probe_cost": 2.8875\n    },\n    "median_mec_to_random_probe_cost_ratio": 0.7793228664697653,\n    "decision": "PRIMARY_PASS_EFFICIENCY_PASS"\n  },\n  "final_policy_rule": "fit once from old train + fresh v1.4 calibration only; qualification test excluded",\n  "hidden_protocol_id": "MEC_V1_4_REACHER_HIDDEN_FINAL_2026_08_30",\n  "runtime_files_sha256": {\n    "MEC_E2E_V1_FROZEN_CONFIG.json": "1aebd294ea9d7b03b0171a90b6d89b4024e9ccb2da997d9b0560591bc7f8cd26",\n    "MEC_E2E_V1_4_ROBUST_SET_COMMIT_DEV_FROZEN_CONFIG.json": "b6905559f63a0bf4bef895baff11a8cff95135ee2ec4ffe0291012814bfd3ec7",\n    "MEC_V1_4_FINAL_POLICY_FREEZE_CONFIG.json": "fb4aeb044945a236754b5020f86af00d627cbe32049e95d1c82de96e63477a77",\n    "MEC_V1_4_REACHER_HIDDEN_FINAL_FROZEN_CONFIG.json": "763882a87e36955b969e3b40f2f403e9ee8f2d0aec6cc148beaf12eb24c2cb4c",\n    "mec_e2e_v1_runner.py": "e127087b0940149c6508df6cba6510c37b0e52fae31971def9affac299bef002",\n    "mec_e2e_v1_dev_full.py": "1f16adb85cd348a47222223a7e9f76d9d3cf6e1dcf414e07b5b865a0efb7eff3",\n    "mec_e2e_v1_1_realistic_dev.py": "421cb39c482463bda4a55748578e651d1618abb3ba6c92775f491f0f38d65a09",\n    "mec_e2e_v1_3_compositional_voi_dev.py": "9f4ec3cbd8dd10e9a3bb46781b734554fedaf49544bae07f5ef7946817f060ac",\n    "mec_e2e_v1_4_robust_set_commit_dev.py": "255104b8623dfd6fd27c44af95fd57f150fbcd0a44ac0fa1a735afeb2a6bb732",\n    "mec_v14_final_policy_freeze.py": "d4fa6cb4acdac9494c1ed8b9915ae97f073c2db6fa6503a774ba57ed003dc709",\n    "mec_v14_reacher_hidden_final.py": "e2dc83bb16bbd174e7f810515fe5bb701c318573539d2e6bbe648f51d3cc0f2a"\n  },\n  "note": "Runtime source/config hashes are frozen before hidden. Notebook hashes are intentionally excluded to avoid self-reference."\n}')
(SRC/'mec_v14_final_policy_freeze.py').write_text("from __future__ import annotations\nimport argparse, hashlib, json, os, pickle, sys, time\nfrom pathlib import Path\nimport numpy as np\nBASE=Path(__file__).resolve().parent; sys.path.insert(0,str(BASE))\nimport mec_e2e_v1_4_robust_set_commit_dev as v14\nPATCH_ID='MEC_V1_4_FINAL_POLICY_FREEZE_R1'\ndef sha256(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()\ndef atomic_json(p,x):\n    p=Path(p); p.parent.mkdir(parents=True,exist_ok=True); t=Path(str(p)+'.tmp'); t.write_text(json.dumps(x,indent=2)); os.replace(t,p)\ndef atomic_pickle(p,x):\n    p=Path(p); p.parent.mkdir(parents=True,exist_ok=True); t=Path(str(p)+'.tmp')\n    with open(t,'wb') as f: pickle.dump(x,f,pickle.HIGHEST_PROTOCOL); f.flush(); os.fsync(f.fileno())\n    os.replace(t,p)\ndef load_pickle(p):\n    with open(p,'rb') as f: return pickle.load(f)\ndef verify_authority(path,names):\n    a=json.loads(Path(path).read_text()); exp=a['runtime_files_sha256']\n    for n in names:\n        p=BASE/n\n        if n not in exp or sha256(p)!=exp[n]: raise RuntimeError('Frozen authority hash mismatch: '+n)\n    return a\n\ndef main():\n    ap=argparse.ArgumentParser(); ap.add_argument('--authority-manifest',required=True); ap.add_argument('--freeze-config',required=True); ap.add_argument('--v14-config',required=True)\n    ap.add_argument('--r2-checkpoint-dir',required=True); ap.add_argument('--v14-checkpoint-dir',required=True); ap.add_argument('--out-dir',required=True)\n    a=ap.parse_args(); fc=json.loads(Path(a.freeze_config).read_text()); cfg=json.loads(Path(a.v14_config).read_text())\n    verify_authority(a.authority_manifest,['mec_v14_final_policy_freeze.py','MEC_V1_4_FINAL_POLICY_FREEZE_CONFIG.json','MEC_E2E_V1_4_ROBUST_SET_COMMIT_DEV_FROZEN_CONFIG.json','mec_e2e_v1_4_robust_set_commit_dev.py','mec_e2e_v1_3_compositional_voi_dev.py','mec_e2e_v1_1_realistic_dev.py','mec_e2e_v1_dev_full.py','mec_e2e_v1_runner.py'])\n    r2=Path(a.r2_checkpoint_dir); v14c=Path(a.v14_checkpoint_dir); out=Path(a.out_dir); out.mkdir(parents=True,exist_ok=True)\n    devp=v14c/'FINAL_RESULT_V14_R1.json'\n    if not devp.exists(): raise FileNotFoundError('Missing completed v1.4 result: '+str(devp))\n    dev=json.loads(devp.read_text()); s=dev.get('summary',{})\n    if s.get('decision') not in ('PRIMARY_PASS_EFFICIENCY_PASS','PRIMARY_PASS_EFFICIENCY_FAIL'): raise RuntimeError('v1.4 development did not authorize final policy freeze')\n    if bool(dev.get('hidden_unsealed',True)): raise RuntimeError('Development result indicates hidden was unsealed')\n    bundlep=out/'MEC_V1_4_FINAL_POLICY_BUNDLE.pkl'; manp=out/'FINAL_POLICY_FREEZE_MANIFEST.json'\n    if bundlep.exists() or manp.exists():\n        if not (bundlep.exists() and manp.exists()): raise RuntimeError('Partial freeze output exists; do not overwrite.')\n        m=json.loads(manp.read_text())\n        if sha256(bundlep)!=m['policy_bundle_sha256']: raise RuntimeError('Frozen policy bundle hash mismatch')\n        print('FINAL POLICY ALREADY FROZEN\\n'+json.dumps(m,indent=2)); return 0\n    seeds=[int(x) for x in cfg['development_world_model_seeds']]\n    oldrec=[]\n    for seed in seeds:\n        rp=r2/f'records_world_model_{seed}_R2.pkl'\n        if not rp.exists(): raise FileNotFoundError(str(rp))\n        oldrec.extend(load_pickle(rp))\n    newrec=[]\n    for seed in seeds:\n        rp=v14c/f'fresh_records_{seed}.pkl'\n        if not rp.exists(): raise FileNotFoundError(str(rp))\n        newrec.extend(load_pickle(rp))\n    Xo,Bo,Wo,Go,Ko=v14.records_to_arrays(oldrec); Xn,Bn,Wn,Gn,Kn=v14.records_to_arrays(newrec)\n    oldtrain=v14.split_old_train(120,int(cfg['old_v13_training']['group_split_seed'])); calg,_=v14.split_fresh(120,int(cfg['fresh_v14_qualification']['split_seed']))\n    train=np.isin(Go,list(oldtrain)); cal=np.isin(Gn,list(calg))\n    models=v14.fit_heads(Xo,Bo,Ko,train,120000,cfg); voi=v14.fit_voi(models,Xo,train,120700); thr=v14.calibrate(models,Xn,Bn,Kn,Wn,cal,seeds)\n    for stage,t in thr.items():\n        if not t.get('calibration_found',False): raise RuntimeError(f'No robust final threshold found for {stage}; hidden NOT authorized')\n        for b in [t.get('pooled',{})]+t.get('per_source_wm',[]):\n            if float(b.get('wrong_ucb',1))>.05+1e-12 or float(b.get('unsafe_ucb',1))>.05+1e-12: raise RuntimeError(f'Final threshold robustness failed for {stage}')\n    bundle={'patch_id':PATCH_ID,'protocol_id':fc['protocol_id'],'development_policy_protocol_id':cfg['protocol_id'],'hidden_unsealed':False,\n      'models':models,'voi':voi,'thresholds':thr,'development_world_model_seeds':seeds,'head_fit_seed':120000,'voi_fit_seed':120700,\n      'old_train_groups':sorted(int(x) for x in oldtrain),'fresh_calibration_groups':sorted(int(x) for x in calg),'v14_qualification_test_groups_used':False}\n    atomic_pickle(bundlep,bundle)\n    manifest={'status':'FINAL_POLICY_FROZEN_HIDDEN_STILL_SEALED','created_unix':time.time(),'patch_id':PATCH_ID,'protocol_id':fc['protocol_id'],\n      'development_decision':s.get('decision'),'development_result_sha256':sha256(devp),'authority_manifest_sha256':sha256(a.authority_manifest),\n      'freeze_config_sha256':sha256(a.freeze_config),'v14_config_sha256':sha256(a.v14_config),'policy_bundle_sha256':sha256(bundlep),'policy_bundle_path':str(bundlep),'thresholds':thr,'hidden_unsealed':False}\n    atomic_json(manp,manifest); atomic_json(out/'FINAL_POLICY_FREEZE_SUMMARY.json',{'status':manifest['status'],'policy_bundle_sha256':manifest['policy_bundle_sha256'],'development_decision':s.get('decision'),'hidden_unsealed':False,'stages':{k:{'tau':v['tau'],'calibration_found':v['calibration_found']} for k,v in thr.items()}})\n    print('\\nFINAL POLICY FREEZE PASS\\n'+json.dumps(manifest,indent=2)); return 0\nif __name__=='__main__': raise SystemExit(main())\n")
(SRC/'mec_v14_reacher_hidden_final.py').write_text("from __future__ import annotations\nimport argparse, hashlib, json, os, pickle, sys, time\nfrom pathlib import Path\nimport numpy as np\nBASE=Path(__file__).resolve().parent; sys.path.insert(0,str(BASE))\nimport mec_e2e_v1_runner as core\nimport mec_e2e_v1_dev_full as dev\nimport mec_e2e_v1_1_realistic_dev as v11\nimport mec_e2e_v1_3_compositional_voi_dev as v13\nimport mec_e2e_v1_4_robust_set_commit_dev as v14\nPATCH_ID='MEC_V1_4_REACHER_HIDDEN_FINAL_R1'; CHECKPOINT_EVERY_EPOCHS=5; ALL=v11.PURE+v11.OOV\ndef sha256(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()\ndef atomic_json(p,x):\n    p=Path(p); p.parent.mkdir(parents=True,exist_ok=True); t=Path(str(p)+'.tmp'); t.write_text(json.dumps(x,indent=2)); os.replace(t,p)\ndef atomic_pickle(p,x):\n    p=Path(p); p.parent.mkdir(parents=True,exist_ok=True); t=Path(str(p)+'.tmp')\n    with open(t,'wb') as f: pickle.dump(x,f,pickle.HIGHEST_PROTOCOL); f.flush(); os.fsync(f.fileno())\n    os.replace(t,p)\ndef load_pickle(p):\n    with open(p,'rb') as f: return pickle.load(f)\ndef verify_authority(path,names):\n    a=json.loads(Path(path).read_text()); exp=a['runtime_files_sha256']\n    for n in names:\n        p=BASE/n\n        if n not in exp or sha256(p)!=exp[n]: raise RuntimeError('Frozen authority hash mismatch: '+n)\n    return a\ndef progress(ck,stage,**kw):\n    x={'stage':stage,'updated_unix':time.time(),**kw}; atomic_json(Path(ck)/'PROGRESS_HIDDEN.json',x); print('\\n'+'='*70+'\\nPROGRESS '+json.dumps(x,indent=2)+'\\n'+'='*70,flush=True)\ndef time_left(d): return None if d is None else d-time.time()\ndef safe_stop(d,reserve=180):\n    x=time_left(d); return False if x is None else x<=reserve\ndef partial_path(ck,s): return Path(ck)/f'hidden_wm_{s}_PARTIAL.pt'\ndef final_path(ck,s): return Path(ck)/f'hidden_world_model_{s}.pt'\ndef move_opt(opt,device):\n    import torch\n    for st in opt.state.values():\n        for k,v in list(st.items()):\n            if torch.is_tensor(v): st[k]=v.to(device)\ndef save_partial(m,opt,g,next_epoch,ck,seed):\n    import torch\n    p=partial_path(ck,seed); t=Path(str(p)+'.tmp'); torch.save({'patch_id':PATCH_ID,'seed':int(seed),'next_epoch':int(next_epoch),'state_dict':{k:v.detach().cpu() for k,v in m.net.state_dict().items()},'optimizer':opt.state_dict(),'generator_state':g.get_state(),'fmu':m.fmu,'fsd':m.fsd,'rmu':m.rmu,'rsd':m.rsd},t); os.replace(t,p)\ndef fit_resumable(m,F,R,epochs,batch,lr,wd,seed,ck,deadline):\n    import torch\n    m.fmu=F.mean(0); m.fsd=F.std(0)+1e-8; m.rmu=R.mean(0); m.rsd=R.std(0)+1e-8\n    Xcpu=torch.tensor((F-m.fmu)/m.fsd,dtype=torch.float32); Ycpu=torch.tensor((R-m.rmu)/m.rsd,dtype=torch.float32); X=Xcpu.to(m.device); Y=Ycpu.to(m.device)\n    opt=torch.optim.AdamW(m.net.parameters(),lr=lr,weight_decay=wd); g=torch.Generator(device='cpu'); g.manual_seed(seed); start=0; pp=partial_path(ck,seed)\n    if pp.exists():\n        o=torch.load(pp,map_location='cpu',weights_only=False)\n        if o.get('patch_id')!=PATCH_ID or int(o.get('seed'))!=int(seed): raise RuntimeError('hidden WM partial mismatch')\n        m.net.load_state_dict(o['state_dict']); opt.load_state_dict(o['optimizer']); move_opt(opt,m.device); g.set_state(o['generator_state']); start=int(o['next_epoch']); m.fmu=np.asarray(o['fmu']); m.fsd=np.asarray(o['fsd']); m.rmu=np.asarray(o['rmu']); m.rsd=np.asarray(o['rsd']); print(f'[resume] hidden WM {seed} epoch {start}/{epochs}',flush=True)\n    for ep in range(start,epochs):\n        order=torch.randperm(len(Xcpu),generator=g)\n        for st in range(0,len(Xcpu),batch):\n            j=order[st:st+batch].to(m.device,non_blocking=True); pred=m.net(X[j]); loss=((pred-Y[j])**2).mean(); opt.zero_grad(set_to_none=True); loss.backward(); opt.step()\n        done=ep+1\n        if done%CHECKPOINT_EVERY_EPOCHS==0 or done==epochs: save_partial(m,opt,g,done,ck,seed); print(f'[hidden WM {seed}] epoch {done}/{epochs} [saved]',flush=True)\n        if safe_stop(deadline,150): save_partial(m,opt,g,done,ck,seed); return None\n    return m\ndef save_final(m,q,norm,ck,seed):\n    import torch\n    p=final_path(ck,seed); t=Path(str(p)+'.tmp'); mu,sd=norm; torch.save({'patch_id':PATCH_ID,'seed':int(seed),'bundle':m.state_dict_bundle(),'quality':q,'nominal_mu':mu,'nominal_sd':sd},t); os.replace(t,p); pp=partial_path(ck,seed); pp.unlink() if pp.exists() else None\ndef load_final(ck,seed,inp,out):\n    import torch\n    p=final_path(ck,seed)\n    if not p.exists(): return None\n    o=torch.load(p,map_location='cpu',weights_only=False); m=dev.FastResidualWM(inp,out,128,3); m.net.load_state_dict(o['bundle']['state_dict']); m.fmu=np.asarray(o['bundle']['fmu']); m.fsd=np.asarray(o['bundle']['fsd']); m.rmu=np.asarray(o['bundle']['rmu']); m.rsd=np.asarray(o['bundle']['rsd']); return m,o['quality'],(np.asarray(o['nominal_mu']),np.asarray(o['nominal_sd']))\ndef load_npz(p):\n    z=np.load(p); return z['X'],z['U'],z['Y'],z['am'].astype(bool)\ndef concrete_seed(root,g,j): return int(np.random.SeedSequence([int(root),int(g),int(j)]).generate_state(1,dtype=np.uint32)[0])\ndef get_scenarios(old,root,n,steps,path,deadline):\n    path=Path(path)\n    if path.exists():\n        s=load_pickle(path); rows=s['rows']; start=int(s['next_group'])\n        if start>=n: return rows,True\n    else: rows=[]; start=0\n    for g in range(start,n):\n        for j,kind in enumerate(ALL): rows.append({'group':g,'kind':kind,'root':int(root),'raw':v11.make_scenario(old,concrete_seed(root,g,j),kind,steps)})\n        if (g+1)%10==0 or g+1==n: atomic_pickle(path,{'next_group':g+1,'rows':rows}); print(f'[hidden scenarios root={root}] {g+1}/{n} [saved]',flush=True)\n        if safe_stop(deadline,180): atomic_pickle(path,{'next_group':g+1,'rows':rows}); return rows,False\n    return rows,True\ndef eval_block(policy,rec,rng_seed):\n    X,B,W,G,K=v14.records_to_arrays(rec); res={}; raw={}\n    for mode,label in [('MEC_VOI','MEC_VOI'),('RANDOM','RANDOM'),('DISCRIM','DISCRIMINATION_HEURISTIC')]:\n        rng=np.random.default_rng(int(rng_seed)+(0 if mode=='MEC_VOI' else 1000 if mode=='RANDOM' else 2000)); outs=[]; costs=[]\n        for i in range(len(B)):\n            a,b,c=v14.policy(policy['models'],policy['thresholds'],policy['voi'],{s:X[s][i] for s in v14.STAGES},mode,rng); outs.append((a,b)); costs.append(c)\n        res[label]=v14.score_outputs(outs,B,costs); raw[label]={'outs':outs,'costs':np.asarray(costs,float)}\n    return {'metrics':res,'ratio':float(res['MEC_VOI']['mean_probe_cost']/max(1e-12,res['RANDOM']['mean_probe_cost'])),'B':B,'raw':raw,'K':K,'G':G}\ndef primary_pass(m): return m['wrong_intervention_risk']<=.05 and m['coverage']>=.70 and m['composite_or_ood_pure_commit']<=.05 and m['composite_or_ood_safe']>=.95\ndef combine(blocks,label):\n    Bs=[]; outs=[]; costs=[]\n    for b in blocks: Bs.append(b['B']); outs.extend(b['raw'][label]['outs']); costs.extend(b['raw'][label]['costs'].tolist())\n    return v14.score_outputs(outs,np.vstack(Bs),np.asarray(costs,float))\n\ndef main():\n    ap=argparse.ArgumentParser(); ap.add_argument('--authority-manifest',required=True); ap.add_argument('--old-config',required=True); ap.add_argument('--hidden-config',required=True); ap.add_argument('--freeze-dir',required=True); ap.add_argument('--r2-checkpoint-dir',required=True); ap.add_argument('--checkpoint-dir',required=True); ap.add_argument('--result-dir',required=True); ap.add_argument('--session-minutes',type=float,default=40); ap.add_argument('--unseal-token',default='NO')\n    a=ap.parse_args(); old=json.loads(Path(a.old_config).read_text()); hc=json.loads(Path(a.hidden_config).read_text())\n    verify_authority(a.authority_manifest,['mec_v14_reacher_hidden_final.py','MEC_V1_4_REACHER_HIDDEN_FINAL_FROZEN_CONFIG.json','MEC_E2E_V1_FROZEN_CONFIG.json','mec_e2e_v1_4_robust_set_commit_dev.py','mec_e2e_v1_3_compositional_voi_dev.py','mec_e2e_v1_1_realistic_dev.py','mec_e2e_v1_dev_full.py','mec_e2e_v1_runner.py'])\n    freeze=Path(a.freeze_dir); r2=Path(a.r2_checkpoint_dir); ck=Path(a.checkpoint_dir); out=Path(a.result_dir); ck.mkdir(parents=True,exist_ok=True); out.mkdir(parents=True,exist_ok=True)\n    if a.unseal_token!=hc['unseal_token']: raise SystemExit('HIDDEN LOCKED: exact unseal token required after final policy freeze.')\n    bundlep=freeze/'MEC_V1_4_FINAL_POLICY_BUNDLE.pkl'; manp=freeze/'FINAL_POLICY_FREEZE_MANIFEST.json'\n    if not bundlep.exists() or not manp.exists(): raise RuntimeError('Final policy freeze missing.')\n    fm=json.loads(manp.read_text())\n    if fm.get('status')!='FINAL_POLICY_FROZEN_HIDDEN_STILL_SEALED' or sha256(bundlep)!=fm.get('policy_bundle_sha256'): raise RuntimeError('Frozen policy identity invalid')\n    policy=load_pickle(bundlep); identity={'policy_bundle_sha256':sha256(bundlep),'hidden_config_sha256':sha256(a.hidden_config),'hidden_runner_sha256':sha256(Path(__file__)),'authority_manifest_sha256':sha256(a.authority_manifest),'protocol_id':hc['protocol_id']}\n    sentinel=ck/'HIDDEN_UNSEAL_SENTINEL.json'\n    if sentinel.exists():\n        oldsent=json.loads(sentinel.read_text())\n        for k,v in identity.items():\n            if oldsent.get(k)!=v: raise RuntimeError('Hidden already unsealed under different frozen identity. STOP.')\n    else: atomic_json(sentinel,{**identity,'unsealed_unix':time.time(),'irreversible':True})\n    started=time.time(); deadline=None if a.session_minutes<=0 else started+a.session_minutes*60\n    print(f'\\n{PATCH_ID}\\nHIDDEN IS NOW UNSEALED UNDER FROZEN HASHES\\nNO POLICY FITTING OR CALIBRATION WILL OCCUR\\n',flush=True)\n    X,U,Y,am=load_npz(r2/'nominal_train.npz'); Xc,Uc,Yc,amc=load_npz(r2/'nominal_calibration.npz'); Xt,Ut,Yt,amt=load_npz(r2/'nominal_test.npz'); F,R=core.make_training_arrays(X,U,Y,am)\n    models=[]; norms=[]; qualities=[]; seeds=[int(x) for x in hc['hidden_world_model_seeds']]\n    for i,seed in enumerate(seeds):\n        progress(ck,'HIDDEN_WORLD_MODEL',index=i+1,total=8,seed=seed); z=load_final(ck,seed,F.shape[1],R.shape[1])\n        if z is None:\n            import torch\n            core.seed_all(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed) if torch.cuda.is_available() else None; m=dev.FastResidualWM(F.shape[1],R.shape[1],128,3); m=fit_resumable(m,F,R,120,old['world_models']['batch_size'],old['world_models']['learning_rate'],old['world_models']['weight_decay'],seed,ck,deadline)\n            if m is None: progress(ck,'SAFE_STOP_DURING_HIDDEN_WM',seed=seed); return 0\n            q=core.evaluate_model(m,Xt,Ut,Yt,am); norm=v13.nominal_norm(m,Xc,Uc,Yc,am); save_final(m,q,norm,ck,seed)\n        else: m,q,norm=z\n        models.append(m); qualities.append(q); norms.append(norm); print(f'[hidden WM {i+1}/8 saved/resumed]',flush=True)\n        if safe_stop(deadline,180): progress(ck,'SAFE_STOP_AFTER_HIDDEN_WM',seed=seed); return 0\n    roots=[int(x) for x in hc['hidden_scenario_seed_roots']]; n=int(hc['hidden_groups_per_seed_root']); steps=int(hc['steps']); scenarios={}\n    for ri,root in enumerate(roots):\n        progress(ck,'HIDDEN_SCENARIOS',index=ri+1,total=len(roots),seed_root=root); rows,done=get_scenarios(old,root,n,steps,ck/f'hidden_scenarios_root_{root}.pkl',deadline); scenarios[root]=rows\n        if not done: progress(ck,'SAFE_STOP_DURING_HIDDEN_SCENARIOS',seed_root=root); return 0\n    blocks=[]\n    for wi,(m,norm,seed) in enumerate(zip(models,norms,seeds)):\n        for ri,root in enumerate(roots):\n            progress(ck,'HIDDEN_FEATURES_AND_EVAL',wm_index=wi+1,root_index=ri+1,hidden_world_model=seed,seed_root=root); rp=ck/f'hidden_records_wm_{seed}_root_{root}.pkl'\n            if rp.exists(): rec=load_pickle(rp)\n            else: rec=v13.build_records([m],[norm],scenarios[root],[seed]); atomic_pickle(rp,rec)\n            bp=ck/f'hidden_block_wm_{seed}_root_{root}.pkl'\n            if bp.exists(): b=load_pickle(bp)\n            else: b=eval_block(policy,rec,305000+wi*100+ri); b['wm']=seed; b['root']=root; atomic_pickle(bp,b)\n            blocks.append(b); print(f'[hidden block {len(blocks)}/{len(seeds)*len(roots)} saved/resumed]',flush=True)\n            if safe_stop(deadline,180): progress(ck,'SAFE_STOP_AFTER_HIDDEN_BLOCK',hidden_world_model=seed,seed_root=root); return 0\n    per_wm=[]\n    for seed in seeds:\n        bb=[b for b in blocks if b['wm']==seed]; m=combine(bb,'MEC_VOI'); r=combine(bb,'RANDOM'); ratio=m['mean_probe_cost']/max(1e-12,r['mean_probe_cost']); per_wm.append({'seed':seed,'MEC_VOI':m,'RANDOM':r,'mec_to_random_probe_cost_ratio':float(ratio),'primary_pass':primary_pass(m),'efficiency_pass':ratio<=.85})\n    per_root=[]\n    for root in roots:\n        bb=[b for b in blocks if b['root']==root]; m=combine(bb,'MEC_VOI'); r=combine(bb,'RANDOM'); per_root.append({'seed_root':root,'MEC_VOI':m,'RANDOM':r,'mec_to_random_probe_cost_ratio':float(m['mean_probe_cost']/max(1e-12,r['mean_probe_cost']))})\n    pooled_m=combine(blocks,'MEC_VOI'); pooled_r=combine(blocks,'RANDOM'); pooled_ratio=pooled_m['mean_probe_cost']/max(1e-12,pooled_r['mean_probe_cost']); med={k:float(np.median([w['MEC_VOI'][k] for w in per_wm])) for k in pooled_m}; medratio=float(np.median([w['mec_to_random_probe_cost_ratio'] for w in per_wm]))\n    qpass=sum(float(q['rmse_ratio'])<=.5 and float(q['normalized_residual_target_mse'])<=.25 for q in qualities); wmpass=sum(w['primary_pass'] for w in per_wm); effpass=sum(w['efficiency_pass'] for w in per_wm); stability=sum(w['MEC_VOI']['wrong_intervention_risk']>.07 for w in per_wm)<=1\n    primary=(qpass>=7 and primary_pass(pooled_m) and wmpass>=6 and primary_pass(med) and stability); efficiency=(medratio<=.85 and effpass>=6); decision='HIDDEN_PRIMARY_PASS_EFFICIENCY_PASS' if primary and efficiency else ('HIDDEN_PRIMARY_PASS_EFFICIENCY_FAIL' if primary else 'HIDDEN_PRIMARY_FAIL')\n    summary={'hidden_unsealed':True,'quality_models_passed':int(qpass),'hidden_wm_primary_passes':int(wmpass),'hidden_wm_efficiency_passes':int(effpass),'pooled_MEC_VOI':pooled_m,'pooled_mec_to_random_probe_cost_ratio':float(pooled_ratio),'median_hidden_wm_MEC_VOI':med,'median_hidden_wm_cost_ratio':medratio,'seed_stability_pass':bool(stability),'decision':decision}\n    atomic_json(out/'MEC_V1_4_REACHER_HIDDEN_FINAL_RESULT.json',{'protocol_id':hc['protocol_id'],'patch_id':PATCH_ID,'frozen_identity':identity,'summary':summary,'quality':qualities,'per_hidden_world_model':per_wm,'per_hidden_scenario_root':per_root}); progress(ck,'COMPLETE',decision=decision); print('\\nFINAL HIDDEN SUMMARY\\n'+json.dumps(summary,indent=2)); return 0\nif __name__=='__main__': raise SystemExit(main())\n")
print('Embedded frozen sources written:', len(list(SRC.iterdir())))

In [ ]:
import subprocess
files=['mec_e2e_v1_runner.py','mec_e2e_v1_dev_full.py','mec_e2e_v1_1_realistic_dev.py','mec_e2e_v1_3_compositional_voi_dev.py','mec_e2e_v1_4_robust_set_commit_dev.py','mec_v14_final_policy_freeze.py']
subprocess.run(['python','-m','py_compile']+[f'/content/mec_final_src/{x}' for x in files],check=True)
print('SOURCE COMPILE PASS')

In [ ]:
!PYTHONUNBUFFERED=1 python -u /content/mec_final_src/mec_v14_final_policy_freeze.py \
  --authority-manifest /content/mec_final_src/MEC_PRE_HIDDEN_AUTHORITY_MANIFEST.json \
  --freeze-config /content/mec_final_src/MEC_V1_4_FINAL_POLICY_FREEZE_CONFIG.json \
  --v14-config /content/mec_final_src/MEC_E2E_V1_4_ROBUST_SET_COMMIT_DEV_FROZEN_CONFIG.json \
  --r2-checkpoint-dir /content/drive/MyDrive/MEC_V1_3_R2_CHECKPOINTS \
  --v14-checkpoint-dir /content/drive/MyDrive/MEC_V1_4_R1_CHECKPOINTS \
  --out-dir /content/drive/MyDrive/MEC_V1_4_FINAL_FREEZE

In [ ]:
from pathlib import Path
import json
p=Path('/content/drive/MyDrive/MEC_V1_4_FINAL_FREEZE/FINAL_POLICY_FREEZE_SUMMARY.json')
print(json.dumps(json.loads(p.read_text()),indent=2) if p.exists() else 'FREEZE NOT COMPLETE')